# song-search -> Geode port build (Windows x86_64)

Cross-compiles `matcool.song-search` for **Geometry Dash 2.2081 (x86_64 / Win64)** on a
Linux kernel, per the Geode docs ("Required C++ Tools" -> Linux) plus matcool's own
`linux-win-cross` gist.

Heavy downloads (xwin splat, Geode SDK, bindings, toolchain) live under `/tmp/work`,
so the kernel output stays small: the mod source, the `.geode`, and `BUILD-LOG.txt`.

1. Install clang / cmake / git / ninja / lld
2. `xwin` splat the MSVC CRT + Windows SDK
3. `matcool/clang-msvc-sdk` CMake toolchain (clang-cl, `x86_64-pc-windows-msvc`)
4. Geode CLI + Geode SDK + 2.2081 bindings
5. Write the mod source, run the pure-C++ logic test natively
6. Configure + build the `.geode`
7. Verify the built DLL architecture and the `mod.json`

CPU only, no GPU/TPU.

In [ ]:
%%bash

set -uo pipefail
mkdir -p /tmp/work
LOG=/kaggle/working/BUILD-LOG.txt
: > "$LOG"
log(){ echo "== $* ==" | tee -a "$LOG"; }

# --- deps as the Geode docs "Required C++ Tools -> Linux" describe ---
log "1. deps: clang clang-tools lld llvm cmake git"
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq >/dev/null 2>&1 || echo "apt update failed" >> "$LOG"
apt-get install -y -qq --no-install-recommends \
    git cmake ninja-build clang clang-tools lld llvm libssl3 \
    unzip xz-utils zip ca-certificates curl wget gpg >/dev/null 2>&1 \
  || echo "apt install partially failed" >> "$LOG"

# The docs note that some distros ship clang cmake that is too old: Geode
# needs Clang 19+ (21+ recommended) and CMake 3.29+, so pull both from the
# upstream repos when the default is too old.
. /etc/os-release
CODENAME=${VERSION_CODENAME:-noble}

CLANG_VER=$(clang --version | grep -oP 'version [0-9]+' | head -1 | grep -oP '[0-9]+')
if [ -z "$CLANG_VER" ] || [ "$CLANG_VER" -lt 19 ]; then
  log "clang ${CLANG_VER:-none} < 19 -> apt.llvm.org llvm-toolchain-$CODENAME-19"
  wget -qO- https://apt.llvm.org/llvm-snapshot.gpg.key 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/llvm.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/llvm.gpg] https://apt.llvm.org/${CODENAME}/ llvm-toolchain-${CODENAME}-19 main" \
    > /etc/apt/sources.list.d/llvm19.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq --no-install-recommends clang-19 lld-19 llvm-19 \
    >/dev/null 2>&1 || echo "apt.llvm.org clang-19 failed" >> "$LOG"
  export PATH=/usr/lib/llvm-19/bin:$PATH
  ln -sf /usr/lib/llvm-19/bin/* /usr/local/bin/ 2>/dev/null || true
  hash -r
fi

CMAKE_VER=$(cmake --version | head -1 | grep -oP '[0-9]+\.[0-9]+' | head -1)
if [ -n "$CMAKE_VER" ] && awk "BEGIN{exit !($CMAKE_VER < 3.29)}"; then
  log "cmake $CMAKE_VER < 3.29 -> apt.kitware.com"
  wget -qO- https://apt.kitware.com/keys/kitware-archive-latest.asc 2>/dev/null \
    | gpg --dearmor -o /usr/share/keyrings/kitware.gpg || true
  echo "deb [signed-by=/usr/share/keyrings/kitware.gpg] https://apt.kitware.com/ubuntu ${CODENAME} main" \
    > /etc/apt/sources.list.d/kitware.list
  apt-get update -qq >/dev/null 2>&1 || true
  apt-get install -y -qq cmake >/dev/null 2>&1 || true
  hash -r
fi

{
  echo "cmake : $(cmake --version | head -1)"
  echo "git   : $(git --version)"
  echo "ninja : $(ninja --version)"
  echo "clang : $(clang --version | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

# Debian/Ubuntu never ship the clang-cl binary, and upstream dropped the
# x86_64 linux tarballs after 18.x. clang selects the CL driver purely from
# argv[0], so a link to clang named "clang-cl" is equivalent.
CLANG_DIR=$(dirname "$(command -v clang)")
if ! command -v clang-cl >/dev/null; then
  log "clang-cl absent -> linking $CLANG_DIR/clang as clang-cl"
  ln -sf "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl" 2>/dev/null \
    || cp -f "$CLANG_DIR/clang" "$CLANG_DIR/clang-cl"
  ln -sf "$CLANG_DIR/clang" /usr/local/bin/clang-cl 2>/dev/null || true
  hash -r
fi

{
  echo "clang : $(clang --version | head -1)"
  echo "clang-cl: $(clang-cl --version 2>&1 | head -1)"
  for t in clang-cl lld-link llvm-lib llvm-nm llvm-mt llvm-rc; do
    if command -v $t >/dev/null; then echo "$t : OK"; else echo "$t : MISSING"; fi
  done
} 2>&1 | tee -a "$LOG"

command -v clang-cl >/dev/null || { echo "FATAL clang-cl missing" | tee -a "$LOG"; exit 2; }
command -v lld-link >/dev/null || { echo "FATAL lld-link missing" | tee -a "$LOG"; exit 2; }
log "DEPS DONE"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "2. xwin splat (MSVC CRT + Windows SDK)"

XWIN_VER=0.10.0
if [ ! -x /tmp/work/xwin/xwin ]; then
  curl -sSL -o /tmp/work/xwin.tar.gz \
    "https://github.com/Jake-Shadle/xwin/releases/download/${XWIN_VER}/xwin-${XWIN_VER}-x86_64-unknown-linux-musl.tar.gz" \
    || { echo "xwin download failed" | tee -a "$LOG"; exit 3; }
  mkdir -p /tmp/work/xwin
  tar -xzf /tmp/work/xwin.tar.gz -C /tmp/work/xwin --strip-components=1
  chmod +x /tmp/work/xwin/xwin
fi
export PATH=/tmp/work/xwin:$PATH
xwin --version | tee -a "$LOG"

# xwin's cache skips already-downloaded payloads, so retrying accumulates
# progress through a flaky connection. Default --arch is x86_64 and the
# default --variant is desktop; debug libs are excluded (Release build).
export XWIN_ACCEPT_LICENSE=1
DONE=0
for attempt in $(seq 1 60); do
  echo "--- xwin splat attempt $attempt/60 ---" >> "$LOG"
  # NB: --accept-license/--arch/--variant/--timeout/--cache-dir are GLOBAL
  # (before the subcommand); --output belongs to the splat subcommand.
  xwin --accept-license --arch x86_64 --variant desktop \
       --timeout 300 --cache-dir /tmp/work/xwin-cache \
       splat --include-debug-libs --output /tmp/work/splat >>"$LOG" 2>&1
  RC=$?
  if [ $RC -eq 0 ]; then
    DONE=1
    log "xwin splat OK on attempt $attempt"
    break
  fi
  echo "attempt $attempt rc=$RC" >> "$LOG"
  sleep 3
done
echo "xwin done=$DONE" | tee -a "$LOG"

echo "--- splat layout ---" | tee -a "$LOG"
du -sh /tmp/work/splat 2>>"$LOG" || true
for d in crt/include crt/lib/x86_64 \
         sdk/include/ucrt sdk/include/um sdk/include/shared sdk/include/winrt \
         sdk/lib/ucrt/x86_64 sdk/lib/um/x86_64; do
  if [ -d "/tmp/work/splat/$d" ]; then
    echo "  OK   $d" | tee -a "$LOG"
  else
    echo "  MISS $d" | tee -a "$LOG"
  fi
done
ls /tmp/work/splat/sdk/include/um/Windows.h >/dev/null 2>&1 \
  && echo "  Windows.h present" | tee -a "$LOG" \
  || echo "  Windows.h MISSING" | tee -a "$LOG"
[ "$DONE" = "1" ] || exit 4

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "3. toolchain + Geode CLI + Geode SDK + bindings"
mkdir -p /tmp/work && cd /tmp/work || exit 5
export PATH=/usr/local/bin:$PATH

if [ ! -d toolchain/.git ]; then
  git clone --depth 1 https://github.com/matcool/clang-msvc-sdk.git toolchain >>"$LOG" 2>&1 \
    || { echo "toolchain clone failed" | tee -a "$LOG"; exit 6; }
fi
ls toolchain | tee -a "$LOG"

GEODE_CLI_VER=v3.9.0
if [ ! -x /tmp/work/geode-cli/geode ]; then
  curl -sSL -o /tmp/work/geode.zip \
    "https://github.com/geode-sdk/cli/releases/download/${GEODE_CLI_VER}/geode-cli-${GEODE_CLI_VER}-linux.zip" \
    || { echo "geode cli download failed" | tee -a "$LOG"; exit 7; }
  rm -rf /tmp/work/geode-cli
  unzip -q -o /tmp/work/geode.zip -d /tmp/work/geode-cli
  chmod +x /tmp/work/geode-cli/geode
fi
export PATH=/tmp/work/geode-cli:$PATH
geode --version | tee -a "$LOG"

# `geode project check` (run by the SDK's CMake) needs a current CLI profile.
# Configure a throwaway one pointing at a scratch dir - we only cross-build
# here, never run the game. It also wants the usual Geode install layout
# present so it can read the list of installed mods.
mkdir -p /tmp/work/gd-scratch/geode/mods
mkdir -p /tmp/work/gd-scratch/geode/config
mkdir -p /tmp/work/gd-scratch/geode/logs
mkdir -p /tmp/work/gd-scratch/geode/unzipped
if ! geode profile list 2>/dev/null | grep -q kaggle; then
  geode profile add --name kaggle /tmp/work/gd-scratch >>"$LOG" 2>&1 || true
fi
geode profile switch kaggle >>"$LOG" 2>&1 || true
echo "profiles:" | tee -a "$LOG"
geode profile list 2>&1 | tee -a "$LOG"

if [ ! -d sdk/.git ]; then
  git clone --depth 1 --branch main https://github.com/geode-sdk/sdk.git sdk >>"$LOG" 2>&1 \
    || { echo "geode sdk clone failed" | tee -a "$LOG"; exit 8; }
fi
geode sdk set-path "/tmp/work/sdk" >>"$LOG" 2>&1 || true
geode sdk version | tee -a "$LOG"
ls sdk/loader/include/link/win64 | tee -a "$LOG"

# The SDK links the Geode loader import library from <sdk>/bin/<version>/.
# `geode sdk install-binaries` fetches it; do it manually if the CLI cannot.
SDK_VER=$(cat sdk/VERSION 2>/dev/null | tr -d '[:space:]')
echo "sdk version: $SDK_VER" | tee -a "$LOG"
if [ -n "$SDK_VER" ] && [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  geode sdk install-binaries >>"$LOG" 2>&1 || true
fi
if [ ! -f "sdk/bin/${SDK_VER}/Geode.lib" ]; then
  log "install-binaries did not provide Geode.lib -> fetching loader zip directly"
  for attempt in $(seq 1 20); do
    curl -sSL -o /tmp/work/geode-win.zip \
      "https://github.com/geode-sdk/geode/releases/download/v${SDK_VER}/geode-v${SDK_VER}-win.zip" && break
    echo "loader zip attempt $attempt failed" >> "$LOG"
    sleep 5
  done
  rm -rf /tmp/work/geode-win && mkdir -p /tmp/work/geode-win
  unzip -q -o /tmp/work/geode-win.zip -d /tmp/work/geode-win || true
  mkdir -p "sdk/bin/${SDK_VER}"
  FOUND=$(find /tmp/work/geode-win -name 'Geode.lib' | head -1)
  if [ -n "$FOUND" ]; then
    cp -f "$FOUND" "sdk/bin/${SDK_VER}/Geode.lib"
    log "installed Geode.lib from $FOUND"
  else
    echo "WARNING: Geode.lib not found in loader zip" | tee -a "$LOG"
    find /tmp/work/geode-win -maxdepth 3 -type f | head -30 | tee -a "$LOG"
  fi
fi
ls -la "sdk/bin/${SDK_VER}/" 2>&1 | tee -a "$LOG"

# Geode generates binding headers at configure time from the bindings repo;
# pre-seed a clone so configure does not fetch it itself
if [ ! -d bindings/.git ]; then
  git clone --depth 1 https://github.com/geode-sdk/bindings.git bindings >>"$LOG" 2>&1 \
    || { echo "bindings clone failed" | tee -a "$LOG"; exit 9; }
fi
ls bindings/bindings/2.2081 | tee -a "$LOG"
log "TOOLCHAIN DONE"

In [ ]:
import base64, json, pathlib
FILES = {
 "mod.json": "ewoJImdlb2RlIjogIjUuMTEuMSIsCgkiZ2QiOiB7CgkJIndpbiI6ICIqIiwKCQkiYW5kcm9pZCI6ICIqIiwKCQkibWFjIjogIioiLAoJCSJpb3MiOiAiKiIKCX0sCgkiaWQiOiAibWF0Y29vbC5zb25nLXNlYXJjaCIsCgkibmFtZSI6ICJTb25nIFNlYXJjaCIsCgkidmVyc2lvbiI6ICJ2MC4yLjgiLAoJImRldmVsb3BlcnMiOiBbCgkJIm5vdGxvdXN5Ym9vayIsCgkJIm1hdGNvb2wiCgldLAoJImRlc2NyaXB0aW9uIjogIlNlYXJjaCB5b3VyIGRvd25sb2FkZWQgc29uZ3Mgd2l0aCBmdXp6eSBtYXRjaGluZywgcGx1cyBKdWtlYm94IE5PTkcgc3VwcG9ydC4iLAoJInRhZ3MiOiBbCgkJIm11c2ljIiwKCQkiaW50ZXJmYWNlIiwKCQkiZW5oYW5jZW1lbnQiCgldLAoJInNldHRpbmdzIjogewoJCSJnZW5lcmFsIjogewoJCQkidHlwZSI6ICJ0aXRsZSIsCgkJCSJuYW1lIjogIkdlbmVyYWwiCgkJfSwKCQkiZGlzYWJsZS1tb2QiOiB7CgkJCSJ0eXBlIjogImJvb2wiLAoJCQkibmFtZSI6ICJEaXNhYmxlIFNvbmcgU2VhcmNoIiwKCQkJImRlc2NyaXB0aW9uIjogIlR1cm4gdGhlIHNlYXJjaCBkaWFsb2cgb2ZmIGVudGlyZWx5IGFuZCByZXN0b3JlIHRoZSBkZWZhdWx0IHNvbmcgYnJvd3Nlci4iLAoJCQkiZGVmYXVsdCI6IGZhbHNlCgkJfSwKCQkianVrZWJveCI6IHsKCQkJInR5cGUiOiAidGl0bGUiLAoJCQkibmFtZSI6ICJKdWtlYm94IE5PTkcgc3VwcG9ydCIKCQl9LAoJCSJub25nLXN5bWJvbHMiOiB7CgkJCSJ0eXBlIjogImJvb2wiLAoJCQkibmFtZSI6ICJTaG93IE5PTkcgc3ltYm9scyIsCgkJCSJkZXNjcmlwdGlvbiI6ICJEcmF3IEp1a2Vib3gncyBwaW4gZGlzYyBvbiBlYWNoIHNvbmcgcm93J3MgdG9wLWxlZnQgY29ybmVyOyBjbGlja2luZyBpdCBvcGVucyBKdWtlYm94J3Mgbm9uZyBwaWNrZXIgZm9yIHRoYXQgc29uZy4gUmVxdWlyZXMgSnVrZWJveCB0byBiZSBpbnN0YWxsZWQ7IGRvZXMgbm90aGluZyB3aXRob3V0IGl0LiIsCgkJCSJkZWZhdWx0IjogdHJ1ZQoJCX0KCX0sCgkicmVzb3VyY2VzIjogewoJCSJzcHJpdGVzaGVldHMiOiB7CgkJCSJTb25nU2VhcmNoU2hlZXQiOiBbCgkJCQkicmVzb3VyY2VzLyoucG5nIgoJCQldCgkJfQoJfSwKCSJsaW5rcyI6IHsKCQkic291cmNlIjogImh0dHBzOi8vZ2l0aHViLmNvbS9ub3Rsb3VzeWJvb2svc29uZy1zZWFyY2gtZ2VvZGUiCgl9LAoJImlzc3VlcyI6IHsKCQkiaW5mbyI6ICJSZXBvcnQgaXNzdWVzIG9uIHRoZSBHaXRIdWIgcmVwb3NpdG9yeSBvZiB0aGlzIHBvcnQuIiwKCQkidXJsIjogImh0dHBzOi8vZ2l0aHViLmNvbS9ub3Rsb3VzeWJvb2svc29uZy1zZWFyY2gtZ2VvZGUvaXNzdWVzIgoJfQp9Cg==",
 "CMakeLists.txt": "Y21ha2VfbWluaW11bV9yZXF1aXJlZChWRVJTSU9OIDMuMjEpCnNldChDTUFLRV9DWFhfU1RBTkRBUkQgMjMpCnNldChDTUFLRV9DWFhfU1RBTkRBUkRfUkVRVUlSRUQgT04pCnNldChDTUFLRV9DWFhfVklTSUJJTElUWV9QUkVTRVQgaGlkZGVuKQoKcHJvamVjdChzb25nLXNlYXJjaCBWRVJTSU9OIDEuMC4wKQoKIyBBZGQgYWxsIHNvdXJjZSBmaWxlcyBpbnNpZGUgc3JjIChyZWN1cnNpdmVseSkKZmlsZShHTE9CX1JFQ1VSU0UgU09VUkNFUyBDT05GSUdVUkVfREVQRU5EUyBzcmMvKi5jcHApCgojIFNldCB1cCB0aGUgbW9kIGJpbmFyeQphZGRfbGlicmFyeSgke1BST0pFQ1RfTkFNRX0gU0hBUkVEICR7U09VUkNFU30pCgppZiAoTk9UIERFRklORUQgRU5We0dFT0RFX1NES30pCiAgICBtZXNzYWdlKEZBVEFMX0VSUk9SICJVbmFibGUgdG8gZmluZCBHZW9kZSBTREshIFBsZWFzZSBkZWZpbmUgR0VPREVfU0RLIGVudmlyb25tZW50IHZhcmlhYmxlIHRvIHBvaW50IHRvIEdlb2RlIikKZWxzZSgpCiAgICBtZXNzYWdlKFNUQVRVUyAiRm91bmQgR2VvZGU6ICRFTlZ7R0VPREVfU0RLfSIpCmVuZGlmKCkKCmFkZF9zdWJkaXJlY3RvcnkoJEVOVntHRU9ERV9TREt9ICR7Q01BS0VfQ1VSUkVOVF9CSU5BUllfRElSfS9nZW9kZSkKCiMgU2V0IHVwIGRlcGVuZGVuY2llcywgcmVzb3VyY2VzLCBhbmQgbGluayBHZW9kZS4Kc2V0dXBfZ2VvZGVfbW9kKCR7UFJPSkVDVF9OQU1FfSkK",
 ".gitignore": "YnVpbGQvCi50b29scy8KKi5nZW9kZQoKIyBLYWdnbGUgYnVpbGQgc2NyYXRjaCBvdXRwdXQgKGxvZ3MgLyBmZXRjaGVkIHBhY2thZ2VzKQoua2FnZ2xlLyovb3V0Ki8KLmthZ2dsZS8qL28qLwoua2FnZ2xlLyovcGtnLwoua2FnZ2xlLyovcHVsbGVkKi8KCiMgS2lsby1pbnN0YWxsZWQgc2tpbGwgY29weSAoc291cmNlIG9mIHRydXRoIGlzIC5rYWdnbGUvc2tpbGxzLykKLmtpbG9jb2RlLwo=",
 "src/main.cpp": "...",
 "src/fts_fuzzy_match.h": "Ly8gTElDRU5TRQovLwovLyAgIFRoaXMgc29mdHdhcmUgaXMgZHVhbC1saWNlbnNlZCB0byB0aGUgcHVibGljIGRvbWFpbiBhbmQgdW5kZXIgdGhlIGZvbGxvd2luZwovLyAgIGxpY2Vuc2U6IHlvdSBhcmUgZ3JhbnRlZCBhIHBlcnBldHVhbCwgaXJyZXZvY2FibGUgbGljZW5zZSB0byBjb3B5LCBtb2RpZnksCi8vICAgcHVibGlzaCwgYW5kIGRpc3RyaWJ1dGUgdGhpcyBmaWxlIGFzIHlvdSBzZWUgZml0LgovLwovLyBWRVJTSU9OCi8vICAgMC4yLjAgICgyMDE3LTAyLTE4KSAgU2NvcmVkIG1hdGNoZXMgcGVyZm9ybSBleGhhdXN0aXZlIHNlYXJjaCBmb3IgYmVzdCBzY29yZQovLyAgIDAuMS4wICAoMjAxNi0wMy0yOCkgIEluaXRpYWwgcmVsZWFzZQovLwovLyBBVVRIT1IKLy8gICBGb3JyZXN0IFNtaXRoCi8vCi8vIE5PVEVTCi8vICAgQ29tcGlsaW5nCi8vICAgICBZb3UgTVVTVCBhZGQgJyNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OJyBiZWZvcmUgaW5jbHVkaW5nIHRoaXMgaGVhZGVyIGluIE9ORSBzb3VyY2UgZmlsZSB0byBjcmVhdGUgaW1wbGVtZW50YXRpb24uCi8vCi8vICAgZnV6enlfbWF0Y2hfc2ltcGxlKC4uLikKLy8gICAgIFJldHVybnMgdHJ1ZSBpZiBlYWNoIGNoYXJhY3RlciBpbiBwYXR0ZXJuIGlzIGZvdW5kIHNlcXVlbnRpYWxseSB3aXRoaW4gc3RyCi8vCi8vICAgZnV6enlfbWF0Y2goLi4uKQovLyAgICAgUmV0dXJucyB0cnVlIGlmIHBhdHRlcm4gaXMgZm91bmQgQU5EIGNhbGN1bGF0ZXMgYSBzY29yZS4KLy8gICAgIFBlcmZvcm1zIGV4aGF1c3RpdmUgc2VhcmNoIHZpYSByZWN1cnNpb24gdG8gZmluZCBhbGwgcG9zc2libGUgbWF0Y2hlcyBhbmQgbWF0Y2ggd2l0aCBoaWdoZXN0IHNjb3JlLgovLyAgICAgU2NvcmVzIHZhbHVlcyBoYXZlIG5vIGludHJpbnNpYyBtZWFuaW5nLiBQb3NzaWJsZSBzY29yZSByYW5nZSBpcyBub3Qgbm9ybWFsaXplZCBhbmQgdmFyaWVzIHdpdGggcGF0dGVybi4KLy8gICAgIFJlY3Vyc2lvbiBpcyBsaW1pdGVkIGludGVybmFsbHkgKGRlZmF1bHQ9MTApIHRvIHByZXZlbnQgZGVnZW5lcmF0ZSBjYXNlcyAocGF0dGVybj0iYWFhYWFhIiBzdHI9ImFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYWFhYSIpCi8vICAgICBVc2VzIHVpbnQ4X3QgZm9yIG1hdGNoIGluZGljZXMuIFRoZXJlZm9yZSBwYXR0ZXJucyBhcmUgbGltaXRlZCB0byAyNTYgY2hhcmFjdGVycy4KLy8gICAgIFNjb3JlIHN5c3RlbSBzaG91bGQgYmUgdHVuZWQgZm9yIFlPVVIgdXNlIGNhc2UuIFdvcmRzLCBzZW50ZW5jZXMsIGZpbGUgbmFtZXMsIG9yIG1ldGhvZCBuYW1lcyBhbGwgcHJlZmVyIGRpZmZlcmVudCB0dW5pbmcuCgojaWZuZGVmIEZUU19GVVpaWV9NQVRDSF9ICiNkZWZpbmUgRlRTX0ZVWlpZX01BVENIX0gKCiNpbmNsdWRlIDxjc3RkaW50PiAvLyB1aW50OF90CiNpbmNsdWRlIDxjdHlwZS5oPiAvLyA6OnRvbG93ZXIsIDo6dG91cHBlcgojaW5jbHVkZSA8Y3N0cmluZz4gLy8gbWVtY3B5CgojaW5jbHVkZSA8Y3N0ZGlvPgoKLy8gUHVibGljIGludGVyZmFjZQpuYW1lc3BhY2UgZnRzIHsKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cik7CiAgICBzdGF0aWMgYm9vbCBmdXp6eV9tYXRjaChjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0ciwgaW50ICYgb3V0U2NvcmUpOwogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlLCB1aW50OF90ICogbWF0Y2hlcywgaW50IG1heE1hdGNoZXMpOwp9CgojaWZkZWYgRlRTX0ZVWlpZX01BVENIX0lNUExFTUVOVEFUSU9OCm5hbWVzcGFjZSBmdHMgewoKICAgIC8vIEZvcndhcmQgZGVjbGFyYXRpb25zIGZvciAicHJpdmF0ZSIgaW1wbGVtZW50YXRpb24KICAgIG5hbWVzcGFjZSBmdXp6eV9pbnRlcm5hbCB7CiAgICAgICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2hfcmVjdXJzaXZlKGNvbnN0IGNoYXIgKiBwYXR0ZXJuLCBjb25zdCBjaGFyICogc3RyLCBpbnQgJiBvdXRTY29yZSwgY29uc3QgY2hhciAqIHN0ckJlZ2luLAogICAgICAgICAgICB1aW50OF90IGNvbnN0ICogc3JjTWF0Y2hlcywgIHVpbnQ4X3QgKiBuZXdNYXRjaGVzLCAgaW50IG1heE1hdGNoZXMsIGludCBuZXh0TWF0Y2gsCiAgICAgICAgICAgIGludCAmIHJlY3Vyc2lvbkNvdW50LCBpbnQgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFB1YmxpYyBpbnRlcmZhY2UKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoX3NpbXBsZShjaGFyIGNvbnN0ICogcGF0dGVybiwgY2hhciBjb25zdCAqIHN0cikgewogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgIHsKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpCiAgICAgICAgICAgICAgICArK3BhdHRlcm47CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgcmV0dXJuICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CiAgICB9CgogICAgc3RhdGljIGJvb2wgZnV6enlfbWF0Y2goY2hhciBjb25zdCAqIHBhdHRlcm4sIGNoYXIgY29uc3QgKiBzdHIsIGludCAmIG91dFNjb3JlKSB7CgogICAgICAgIHVpbnQ4X3QgbWF0Y2hlc1syNTZdOwogICAgICAgIHJldHVybiBmdXp6eV9tYXRjaChwYXR0ZXJuLCBzdHIsIG91dFNjb3JlLCBtYXRjaGVzLCBzaXplb2YobWF0Y2hlcykpOwogICAgfQoKICAgIHN0YXRpYyBib29sIGZ1enp5X21hdGNoKGNoYXIgY29uc3QgKiBwYXR0ZXJuLCBjaGFyIGNvbnN0ICogc3RyLCBpbnQgJiBvdXRTY29yZSwgdWludDhfdCAqIG1hdGNoZXMsIGludCBtYXhNYXRjaGVzKSB7CiAgICAgICAgaW50IHJlY3Vyc2lvbkNvdW50ID0gMDsKICAgICAgICBpbnQgcmVjdXJzaW9uTGltaXQgPSAxMDsKCiAgICAgICAgcmV0dXJuIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyLCBvdXRTY29yZSwgc3RyLCBudWxscHRyLCBtYXRjaGVzLCBtYXhNYXRjaGVzLCAwLCByZWN1cnNpb25Db3VudCwgcmVjdXJzaW9uTGltaXQpOwogICAgfQoKICAgIC8vIFByaXZhdGUgaW1wbGVtZW50YXRpb24KICAgIHN0YXRpYyBib29sIGZ1enp5X2ludGVybmFsOjpmdXp6eV9tYXRjaF9yZWN1cnNpdmUoY29uc3QgY2hhciAqIHBhdHRlcm4sIGNvbnN0IGNoYXIgKiBzdHIsIGludCAmIG91dFNjb3JlLAogICAgICAgIGNvbnN0IGNoYXIgKiBzdHJCZWdpbiwgdWludDhfdCBjb25zdCAqIHNyY01hdGNoZXMsIHVpbnQ4X3QgKiBtYXRjaGVzLCBpbnQgbWF4TWF0Y2hlcywKICAgICAgICBpbnQgbmV4dE1hdGNoLCBpbnQgJiByZWN1cnNpb25Db3VudCwgaW50IHJlY3Vyc2lvbkxpbWl0KQogICAgewogICAgICAgIC8vIENvdW50IHJlY3Vyc2lvbnMKICAgICAgICArK3JlY3Vyc2lvbkNvdW50OwogICAgICAgIGlmIChyZWN1cnNpb25Db3VudCA+PSByZWN1cnNpb25MaW1pdCkKICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAvLyBEZXRlY3QgZW5kIG9mIHN0cmluZ3MKICAgICAgICBpZiAoKnBhdHRlcm4gPT0gJ1wwJyB8fCAqc3RyID09ICdcMCcpCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKCiAgICAgICAgLy8gUmVjdXJzaW9uIHBhcmFtcwogICAgICAgIGJvb2wgcmVjdXJzaXZlTWF0Y2ggPSBmYWxzZTsKICAgICAgICB1aW50OF90IGJlc3RSZWN1cnNpdmVNYXRjaGVzWzI1Nl07CiAgICAgICAgaW50IGJlc3RSZWN1cnNpdmVTY29yZSA9IDA7CgogICAgICAgIC8vIExvb3AgdGhyb3VnaCBwYXR0ZXJuIGFuZCBzdHIgbG9va2luZyBmb3IgYSBtYXRjaAogICAgICAgIGJvb2wgZmlyc3RfbWF0Y2ggPSB0cnVlOwogICAgICAgIHdoaWxlICgqcGF0dGVybiAhPSAnXDAnICYmICpzdHIgIT0gJ1wwJykgewoKICAgICAgICAgICAgLy8gRm91bmQgbWF0Y2gKICAgICAgICAgICAgaWYgKHRvbG93ZXIoKnBhdHRlcm4pID09IHRvbG93ZXIoKnN0cikpIHsKCiAgICAgICAgICAgICAgICAvLyBTdXBwbGllZCBtYXRjaGVzIGJ1ZmZlciB3YXMgdG9vIHNob3J0CiAgICAgICAgICAgICAgICBpZiAobmV4dE1hdGNoID49IG1heE1hdGNoZXMpCiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIGZhbHNlOwoKICAgICAgICAgICAgICAgIC8vICJDb3B5LW9uLVdyaXRlIiBzcmNNYXRjaGVzIGludG8gbWF0Y2hlcwogICAgICAgICAgICAgICAgaWYgKGZpcnN0X21hdGNoICYmIHNyY01hdGNoZXMpIHsKICAgICAgICAgICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgc3JjTWF0Y2hlcywgbmV4dE1hdGNoKTsKICAgICAgICAgICAgICAgICAgICBmaXJzdF9tYXRjaCA9IGZhbHNlOwogICAgICAgICAgICAgICAgfQoKICAgICAgICAgICAgICAgIC8vIFJlY3Vyc2l2ZSBjYWxsIHRoYXQgInNraXBzIiB0aGlzIG1hdGNoCiAgICAgICAgICAgICAgICB1aW50OF90IHJlY3Vyc2l2ZU1hdGNoZXNbMjU2XTsKICAgICAgICAgICAgICAgIGludCByZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgICAgIGlmIChmdXp6eV9tYXRjaF9yZWN1cnNpdmUocGF0dGVybiwgc3RyICsgMSwgcmVjdXJzaXZlU2NvcmUsIHN0ckJlZ2luLCBtYXRjaGVzLCByZWN1cnNpdmVNYXRjaGVzLCBzaXplb2YocmVjdXJzaXZlTWF0Y2hlcyksIG5leHRNYXRjaCwgcmVjdXJzaW9uQ291bnQsIHJlY3Vyc2lvbkxpbWl0KSkgewoKICAgICAgICAgICAgICAgICAgICAvLyBQaWNrIGJlc3QgcmVjdXJzaXZlIHNjb3JlCiAgICAgICAgICAgICAgICAgICAgaWYgKCFyZWN1cnNpdmVNYXRjaCB8fCByZWN1cnNpdmVTY29yZSA+IGJlc3RSZWN1cnNpdmVTY29yZSkgewogICAgICAgICAgICAgICAgICAgICAgICBtZW1jcHkoYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIHJlY3Vyc2l2ZU1hdGNoZXMsIDI1Nik7CiAgICAgICAgICAgICAgICAgICAgICAgIGJlc3RSZWN1cnNpdmVTY29yZSA9IHJlY3Vyc2l2ZVNjb3JlOwogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICAgICByZWN1cnNpdmVNYXRjaCA9IHRydWU7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQWR2YW5jZQogICAgICAgICAgICAgICAgbWF0Y2hlc1tuZXh0TWF0Y2grK10gPSAodWludDhfdCkoc3RyIC0gc3RyQmVnaW4pOwogICAgICAgICAgICAgICAgKytwYXR0ZXJuOwogICAgICAgICAgICB9CiAgICAgICAgICAgICsrc3RyOwogICAgICAgIH0KCiAgICAgICAgLy8gRGV0ZXJtaW5lIGlmIGZ1bGwgcGF0dGVybiB3YXMgbWF0Y2hlZAogICAgICAgIGJvb2wgbWF0Y2hlZCA9ICpwYXR0ZXJuID09ICdcMCcgPyB0cnVlIDogZmFsc2U7CgogICAgICAgIC8vIENhbGN1bGF0ZSBzY29yZQogICAgICAgIGlmIChtYXRjaGVkKSB7CiAgICAgICAgICAgIGNvbnN0IGludCBzZXF1ZW50aWFsX2JvbnVzID0gMTU7ICAgICAgICAgICAgLy8gYm9udXMgZm9yIGFkamFjZW50IG1hdGNoZXMKICAgICAgICAgICAgY29uc3QgaW50IHNlcGFyYXRvcl9ib251cyA9IDMwOyAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBvY2N1cnMgYWZ0ZXIgYSBzZXBhcmF0b3IKICAgICAgICAgICAgY29uc3QgaW50IGNhbWVsX2JvbnVzID0gMzA7ICAgICAgICAgICAgICAgICAvLyBib251cyBpZiBtYXRjaCBpcyB1cHBlcmNhc2UgYW5kIHByZXYgaXMgbG93ZXIKICAgICAgICAgICAgY29uc3QgaW50IGZpcnN0X2xldHRlcl9ib251cyA9IDE1OyAgICAgICAgICAvLyBib251cyBpZiB0aGUgZmlyc3QgbGV0dGVyIGlzIG1hdGNoZWQKCiAgICAgICAgICAgIGNvbnN0IGludCBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTU7ICAgICAgLy8gcGVuYWx0eSBhcHBsaWVkIGZvciBldmVyeSBsZXR0ZXIgaW4gc3RyIGJlZm9yZSB0aGUgZmlyc3QgbWF0Y2gKICAgICAgICAgICAgY29uc3QgaW50IG1heF9sZWFkaW5nX2xldHRlcl9wZW5hbHR5ID0gLTE1OyAvLyBtYXhpbXVtIHBlbmFsdHkgZm9yIGxlYWRpbmcgbGV0dGVycwogICAgICAgICAgICBjb25zdCBpbnQgdW5tYXRjaGVkX2xldHRlcl9wZW5hbHR5ID0gLTE7ICAgIC8vIHBlbmFsdHkgZm9yIGV2ZXJ5IGxldHRlciB0aGF0IGRvZXNuJ3QgbWF0dGVyCgogICAgICAgICAgICAvLyBJdGVyYXRlIHN0ciB0byBlbmQKICAgICAgICAgICAgd2hpbGUgKCpzdHIgIT0gJ1wwJykKICAgICAgICAgICAgICAgICsrc3RyOwoKICAgICAgICAgICAgLy8gSW5pdGlhbGl6ZSBzY29yZQogICAgICAgICAgICBvdXRTY29yZSA9IDEwMDsKCiAgICAgICAgICAgIC8vIEFwcGx5IGxlYWRpbmcgbGV0dGVyIHBlbmFsdHkKICAgICAgICAgICAgaW50IHBlbmFsdHkgPSBsZWFkaW5nX2xldHRlcl9wZW5hbHR5ICogbWF0Y2hlc1swXTsKICAgICAgICAgICAgaWYgKHBlbmFsdHkgPCBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eSkKICAgICAgICAgICAgICAgIHBlbmFsdHkgPSBtYXhfbGVhZGluZ19sZXR0ZXJfcGVuYWx0eTsKICAgICAgICAgICAgb3V0U2NvcmUgKz0gcGVuYWx0eTsKCiAgICAgICAgICAgIC8vIEFwcGx5IHVubWF0Y2hlZCBwZW5hbHR5CiAgICAgICAgICAgIGludCB1bm1hdGNoZWQgPSAoaW50KShzdHIgLSBzdHJCZWdpbikgLSBuZXh0TWF0Y2g7CiAgICAgICAgICAgIG91dFNjb3JlICs9IHVubWF0Y2hlZF9sZXR0ZXJfcGVuYWx0eSAqIHVubWF0Y2hlZDsKCiAgICAgICAgICAgIC8vIEFwcGx5IG9yZGVyaW5nIGJvbnVzZXMKICAgICAgICAgICAgZm9yIChpbnQgaSA9IDA7IGkgPCBuZXh0TWF0Y2g7ICsraSkgewogICAgICAgICAgICAgICAgdWludDhfdCBjdXJySWR4ID0gbWF0Y2hlc1tpXTsKCiAgICAgICAgICAgICAgICBpZiAoaSA+IDApIHsKICAgICAgICAgICAgICAgICAgICB1aW50OF90IHByZXZJZHggPSBtYXRjaGVzW2kgLSAxXTsKCiAgICAgICAgICAgICAgICAgICAgLy8gU2VxdWVudGlhbAogICAgICAgICAgICAgICAgICAgIGlmIChjdXJySWR4ID09IChwcmV2SWR4ICsgMSkpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcXVlbnRpYWxfYm9udXM7CiAgICAgICAgICAgICAgICB9CgogICAgICAgICAgICAgICAgLy8gQ2hlY2sgZm9yIGJvbnVzZXMgYmFzZWQgb24gbmVpZ2hib3IgY2hhcmFjdGVyIHZhbHVlCiAgICAgICAgICAgICAgICBpZiAoY3VycklkeCA+IDApIHsKICAgICAgICAgICAgICAgICAgICAvLyBDYW1lbCBjYXNlCiAgICAgICAgICAgICAgICAgICAgY2hhciBuZWlnaGJvciA9IHN0ckJlZ2luW2N1cnJJZHggLSAxXTsKICAgICAgICAgICAgICAgICAgICBjaGFyIGN1cnIgPSBzdHJCZWdpbltjdXJySWR4XTsKICAgICAgICAgICAgICAgICAgICBpZiAoOjppc2xvd2VyKG5laWdoYm9yKSAmJiA6OmlzdXBwZXIoY3VycikpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGNhbWVsX2JvbnVzOwoKICAgICAgICAgICAgICAgICAgICAvLyBTZXBhcmF0b3IKICAgICAgICAgICAgICAgICAgICBib29sIG5laWdoYm9yU2VwYXJhdG9yID0gbmVpZ2hib3IgPT0gJ18nIHx8IG5laWdoYm9yID09ICcgJzsKICAgICAgICAgICAgICAgICAgICBpZiAobmVpZ2hib3JTZXBhcmF0b3IpCiAgICAgICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IHNlcGFyYXRvcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgIGVsc2UgewogICAgICAgICAgICAgICAgICAgIC8vIEZpcnN0IGxldHRlcgogICAgICAgICAgICAgICAgICAgIG91dFNjb3JlICs9IGZpcnN0X2xldHRlcl9ib251czsKICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgfQogICAgICAgIH0KCiAgICAgICAgLy8gUmV0dXJuIGJlc3QgcmVzdWx0CiAgICAgICAgaWYgKHJlY3Vyc2l2ZU1hdGNoICYmICghbWF0Y2hlZCB8fCBiZXN0UmVjdXJzaXZlU2NvcmUgPiBvdXRTY29yZSkpIHsKICAgICAgICAgICAgLy8gUmVjdXJzaXZlIHNjb3JlIGlzIGJldHRlciB0aGFuICJ0aGlzIgogICAgICAgICAgICBtZW1jcHkobWF0Y2hlcywgYmVzdFJlY3Vyc2l2ZU1hdGNoZXMsIG1heE1hdGNoZXMpOwogICAgICAgICAgICBvdXRTY29yZSA9IGJlc3RSZWN1cnNpdmVTY29yZTsKICAgICAgICAgICAgcmV0dXJuIHRydWU7CiAgICAgICAgfQogICAgICAgIGVsc2UgaWYgKG1hdGNoZWQpIHsKICAgICAgICAgICAgLy8gInRoaXMiIHNjb3JlIGlzIGJldHRlciB0aGFuIHJlY3Vyc2l2ZQogICAgICAgICAgICByZXR1cm4gdHJ1ZTsKICAgICAgICB9CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIC8vIG5vIG1hdGNoCiAgICAgICAgICAgIHJldHVybiBmYWxzZTsKICAgICAgICB9CiAgICB9Cn0gLy8gbmFtZXNwYWNlIGZ0cwoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9JTVBMRU1FTlRBVElPTgoKI2VuZGlmIC8vIEZUU19GVVpaWV9NQVRDSF9ICg==",
 "tests/logic_test.cpp": "Ly8gU3RhbmRhbG9uZSBsb2dpYyB0ZXN0IGZvciB0aGUgc29uZy1zZWFyY2ggcG9ydC4KLy8gTWlycm9ycyB0aGUgZnV6enkgc2VhcmNoICsgbGFiZWwtZml0ICsgc2Nyb2xsLXdpbmRvdyBsb2dpYyBpbiBzcmMvbWFpbi5jcHAKLy8gd2l0aG91dCBHZW9kZSBoZWFkZXJzLCBzbyBpdCBjYW4gcnVuIG9uIHBsYWluIEdDQy4KI2RlZmluZSBGVFNfRlVaWllfTUFUQ0hfSU1QTEVNRU5UQVRJT04KI2luY2x1ZGUgImZ0c19mdXp6eV9tYXRjaC5oIgoKI2luY2x1ZGUgPGFsZ29yaXRobT4KI2luY2x1ZGUgPGNhc3NlcnQ+CiNpbmNsdWRlIDxjbWF0aD4KI2luY2x1ZGUgPGlvc3RyZWFtPgojaW5jbHVkZSA8c3N0cmVhbT4KI2luY2x1ZGUgPHN0cmluZz4KI2luY2x1ZGUgPHZlY3Rvcj4KCnN0cnVjdCBGYWtlU29uZyB7IGludCBpZDsgc3RkOjpzdHJpbmcgbmFtZTsgc3RkOjpzdHJpbmcgYXJ0aXN0OyB9OwoKc3RhdGljIHN0ZDo6dmVjdG9yPHN0ZDo6cGFpcjxpbnQsIEZha2VTb25nPj4gZG9fc2VhcmNoKAogICAgY29uc3Qgc3RkOjp2ZWN0b3I8c3RkOjpwYWlyPHN0ZDo6c3RyaW5nLCBGYWtlU29uZz4+JiBkb3dubG9hZGVkLAogICAgY29uc3Qgc3RkOjpzdHJpbmcmIGlucHV0KSB7CiAgICBzdGQ6OnZlY3RvcjxzdGQ6OnBhaXI8aW50LCBGYWtlU29uZz4+IHNvbmdzOwogICAgc29uZ3MucmVzZXJ2ZShkb3dubG9hZGVkLnNpemUoKSk7CiAgICBjb25zdCBib29sIGVtcHR5X2lucHV0ID0gaW5wdXQuZW1wdHkoKTsKICAgIGZvciAoY29uc3QgYXV0byYgW2luZGV4X3N0ciwgc29uZ10gOiBkb3dubG9hZGVkKSB7CiAgICAgICAgaW50IHNjb3JlID0gMDsKICAgICAgICBpZiAoIWVtcHR5X2lucHV0ICYmICFmdHM6OmZ1enp5X21hdGNoKGlucHV0LmNfc3RyKCksIGluZGV4X3N0ci5jX3N0cigpLCBzY29yZSkpIGNvbnRpbnVlOwogICAgICAgIHNvbmdzLnB1c2hfYmFjayh7c2NvcmUsIHNvbmd9KTsKICAgIH0KICAgIGlmICghZW1wdHlfaW5wdXQgJiYgc29uZ3Muc2l6ZSgpID4gMSkgewogICAgICAgIHN0ZDo6c29ydChzb25ncy5iZWdpbigpLCBzb25ncy5lbmQoKSwgWyZdKGNvbnN0IGF1dG8mIGEsIGNvbnN0IGF1dG8mIGIpIHsKICAgICAgICAgICAgcmV0dXJuIGEuZmlyc3QgPiBiLmZpcnN0OwogICAgICAgIH0pOwogICAgfQogICAgcmV0dXJuIHNvbmdzOwp9CgppbnQgbWFpbigpIHsKICAgIHN0ZDo6dmVjdG9yPHN0ZDo6cGFpcjxzdGQ6OnN0cmluZywgRmFrZVNvbmc+PiBkb3dubG9hZGVkID0gewogICAgICAgIHsiU3RlcmVvIE1hZG5lc3MgRm9yZXZlciIsIHsxLCAiU3RlcmVvIE1hZG5lc3MiLCAiRm9yZXZlciJ9fSwKICAgICAgICB7IkJhY2sgb24gVHJhY2sgREpWSSIsIHsyLCAiQmFjayBvbiBUcmFjayIsICJESlZJIn19LAogICAgICAgIHsiUG9sYXJnZWlzdCBTdGVwIiwgezMsICJQb2xhcmdlaXN0IiwgIlN0ZXAifX0sCiAgICAgICAgeyJEcnkgT3V0IERKVkkiLCB7NCwgIkRyeSBPdXQiLCAiREpWSSJ9fSwKICAgICAgICB7IkJhc2UgQWZ0ZXIgQmFzZSBESlZJIiwgezUsICJCYXNlIEFmdGVyIEJhc2UiLCAiREpWSSJ9fSwKICAgIH07CgogICAgLy8gMS4gZW1wdHkgaW5wdXQgcmV0dXJucyBldmVyeXRoaW5nLCB1bnNvcnRlZAogICAgYXV0byBhbGwgPSBkb19zZWFyY2goZG93bmxvYWRlZCwgIiIpOwogICAgYXNzZXJ0KGFsbC5zaXplKCkgPT0gNSk7CgogICAgLy8gMi4gZXhhY3QtaXNoIHF1ZXJ5IHJhbmtzIHRoZSByaWdodCBzb25nIGZpcnN0CiAgICBhdXRvIGRqdmkgPSBkb19zZWFyY2goZG93bmxvYWRlZCwgImRqdmkiKTsKICAgIGFzc2VydChkanZpLnNpemUoKSA9PSAzKTsKICAgIGZvciAoYXV0byYgW3Njb3JlLCBzXSA6IGRqdmkpICh2b2lkKXNjb3JlLCBhc3NlcnQocy5hcnRpc3QgPT0gIkRKVkkiKTsKCiAgICAvLyAzLiBmdXp6eSBxdWVyeSBtYXRjaGVzIGFjcm9zcyBuYW1lK2FydGlzdCBibG9iCiAgICBhdXRvIHBvbGFyID0gZG9fc2VhcmNoKGRvd25sb2FkZWQsICJwb2xyIik7CiAgICBhc3NlcnQocG9sYXIuc2l6ZSgpID09IDEgJiYgcG9sYXJbMF0uc2Vjb25kLmlkID09IDMpOwoKICAgIC8vIDQuIG5vIG1hdGNoIC0+IGVtcHR5CiAgICBhc3NlcnQoZG9fc2VhcmNoKGRvd25sb2FkZWQsICJ6enpxcXEiKS5lbXB0eSgpKTsKCiAgICAvLyA1LiBzY3JvbGwgY2xhbXA6IG1pcnJvcnMgdXBkYXRlX3Njcm9sbF9lbGVtZW50cyBjbGFtcGluZwogICAgY29uc3RleHByIGZsb2F0IGhlaWdodCA9IDU1LmY7IC8vIHdpZGdldF9oZWlnaHQgKyBzcGFjaW5nCiAgICBjb25zdGV4cHIgaW50IG5fd2lkZ2V0cyA9IDY7CiAgICBhdXRvIGNsYW1wX3Njcm9sbCA9IFsmXShzaXplX3QgY291bnQsIGZsb2F0IHkpIHsKICAgICAgICBjb25zdCBhdXRvIG0gPSBoZWlnaHQgKiAoc3RhdGljX2Nhc3Q8aW50Pihjb3VudCkgLSBuX3dpZGdldHMgKyAzKTsKICAgICAgICBpZiAobSA8IDApIHJldHVybiAwLmY7CiAgICAgICAgcmV0dXJuIHN0ZDo6Y2xhbXAoeSwgMC5mLCBtKTsKICAgIH07CiAgICBhc3NlcnQoY2xhbXBfc2Nyb2xsKDAsIDk5OS5mKSA9PSAwLmYpOyAgIC8vIGVtcHR5IGxpc3QgcGlucyB0byAwCiAgICBhc3NlcnQoY2xhbXBfc2Nyb2xsKDMsIDk5OS5mKSA9PSAwLmYpOyAgIC8vIHNob3J0IGxpc3QgcGlucyB0byAwCiAgICBhc3NlcnQoY2xhbXBfc2Nyb2xsKDIwLCAtNTAuZikgPT0gMC5mKTsgIC8vIG5lZ2F0aXZlIHBpbnMgdG8gMAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgyMCwgMWU5ZikgPT0gaGVpZ2h0ICogKDIwIC0gbl93aWRnZXRzICsgMykpOyAvLyBjYXBzIGF0IG1heAogICAgYXNzZXJ0KGNsYW1wX3Njcm9sbCgyMCwgMTAwLmYpID09IDEwMC5mKTsgLy8gaW4tcmFuZ2UgdW50b3VjaGVkCgogICAgLy8gNi4gcmVzdWx0IGxhYmVsIHBsdXJhbGl6YXRpb24KICAgIGF1dG8gbGFiZWwgPSBbXShzaXplX3QgbikgewogICAgICAgIHN0ZDo6c3RyaW5nc3RyZWFtIHM7IHMgPDwgbiA8PCAiIHJlc3VsdCI7IGlmIChuICE9IDEpIHMgPDwgInMiOyByZXR1cm4gcy5zdHIoKTsKICAgIH07CiAgICBhc3NlcnQobGFiZWwoMCkgPT0gIjAgcmVzdWx0cyIgJiYgbGFiZWwoMSkgPT0gIjEgcmVzdWx0IiAmJiBsYWJlbCg1KSA9PSAiNSByZXN1bHRzIik7CgogICAgLy8gNy4gZXhwby1vdXQgZWFzaW5nOiByZW5kZXJzIGVhc2UgdG93YXJkIGEgdGFyZ2V0IHRoZSBzYW1lIHdheSB0aGUgbW9kCiAgICAvLyAgICBkb2VzLCBtdXN0IGJlIG1vbm90b25pYywgZnJhbWVyYXRlLWNvcnJlY3QsIGFuZCBsYW5kIGV4YWN0bHkgb24gdGFyZ2V0CiAgICBhdXRvIGVhc2VTdGVwID0gW10oZmxvYXQgY3VycmVudCwgZmxvYXQgdGFyZ2V0LCBmbG9hdCBkdCkgLT4gZmxvYXQgewogICAgICAgIGNvbnN0IGZsb2F0IGZhY3RvciA9IDEuZiAtIHN0ZDo6ZXhwKC0xOC5mICogZHQpOwogICAgICAgIHJldHVybiBjdXJyZW50ICsgKHRhcmdldCAtIGN1cnJlbnQpICogZmFjdG9yOwogICAgfTsKICAgIHsKICAgICAgICBmbG9hdCBwb3MgPSAwLmYsIHRhcmdldCA9IDMwMC5mOwogICAgICAgIGZsb2F0IHByZXYgPSBwb3M7CiAgICAgICAgaW50IHN0ZXBzID0gMDsKICAgICAgICB3aGlsZSAoc3RkOjphYnModGFyZ2V0IC0gcG9zKSA+IDAuMDFmICYmIHN0ZXBzIDwgMTAwMDAwKSB7CiAgICAgICAgICAgIHBvcyA9IGVhc2VTdGVwKHBvcywgdGFyZ2V0LCAxLmYgLyA2MC5mKTsKICAgICAgICAgICAgYXNzZXJ0KHBvcyA+PSBwcmV2IC0gMWUtNmYpOyAgIC8vIG5ldmVyIG1vdmVzIGJhY2t3YXJkcwogICAgICAgICAgICBhc3NlcnQocG9zIDw9IHRhcmdldCArIDFlLTZmKTsgLy8gbmV2ZXIgb3ZlcnNob290cyAoZXhwbyBvdXQpCiAgICAgICAgICAgIHByZXYgPSBwb3M7CiAgICAgICAgICAgICsrc3RlcHM7CiAgICAgICAgfQogICAgICAgIGFzc2VydChzdGQ6OmFicyhwb3MgLSB0YXJnZXQpIDwgMC4wMWYpOwogICAgICAgIC8vIGEgMzBmcHMgcnVuIHNob3VsZCBuZWVkIGZld2VyIHN0ZXBzIHRoYW4gdGhlIHNhbWUgZGlzdGFuY2UgYXQgNjBmcHMKICAgICAgICBmbG9hdCBwb3MzMCA9IDAuZjsKICAgICAgICBpbnQgc3RlcHMzMCA9IDA7CiAgICAgICAgd2hpbGUgKHN0ZDo6YWJzKHRhcmdldCAtIHBvczMwKSA+IDAuMDFmICYmIHN0ZXBzMzAgPCAxMDAwMDApIHsKICAgICAgICAgICAgcG9zMzAgPSBlYXNlU3RlcChwb3MzMCwgdGFyZ2V0LCAxLmYgLyAzMC5mKTsKICAgICAgICAgICAgKytzdGVwczMwOwogICAgICAgIH0KICAgICAgICBhc3NlcnQoc3RlcHMzMCA8IHN0ZXBzKTsKICAgICAgICBhc3NlcnQoc3RkOjphYnMocG9zMzAgLSB0YXJnZXQpIDwgMC4wMWYpOwogICAgICAgIHN0ZDo6Y291dCA8PCAiICBlYXNpbmc6IDYwZnBzICIgPDwgc3RlcHMgPDwgIiBzdGVwcywgMzBmcHMgIiA8PCBzdGVwczMwIDw8ICIgc3RlcHNcbiI7CiAgICB9CgogICAgLy8gOC4gbGFiZWwgZml0OiBtaXJyb3JzIFNpbXBsZVNvbmdXaWRnZXQ6OmZpdExhYmVsIC0gc2hyaW5rIGZyb20gdGhlIGJhc2UKICAgIC8vICAgIHNjYWxlIGRvd24gdG93YXJkIG1pblNjYWxlLCB0aGVuIHRydW5jYXRlIHdpdGggIi4uLiIgc28gdGV4dCBjYW4KICAgIC8vICAgIG5ldmVyIHJ1biB1bmRlciB0aGUgcm93J3MgYnV0dG9ucwogICAgewogICAgICAgIGF1dG8gd2lkdGggPSBbXShjb25zdCBzdGQ6OnN0cmluZyYgcykgeyByZXR1cm4gMTAuZiAqIHMuc2l6ZSgpOyB9OyAvLyBmYWtlIGZvbnQ6IDEwcHgvY2hhcgogICAgICAgIGF1dG8gZml0ID0gWyZdKGNvbnN0IHN0ZDo6c3RyaW5nJiB0ZXh0LCBmbG9hdCBiYXNlLCBmbG9hdCBtYXhXLCBmbG9hdCBtaW5TKSAtPiBzdGQ6OnN0cmluZyB7CiAgICAgICAgICAgIGNvbnN0IGZsb2F0IHNjYWxlID0gc3RkOjptaW4oYmFzZSwgbWF4VyAvIHdpZHRoKHRleHQpKTsKICAgICAgICAgICAgaWYgKHNjYWxlID49IG1pblMpIHJldHVybiB0ZXh0OyAvLyBzaG93biB3aG9sZSAocG9zc2libHkgc2hydW5rKQogICAgICAgICAgICBzaXplX3QgbG8gPSAwLCBoaSA9IHRleHQuc2l6ZSgpOwogICAgICAgICAgICBzdGQ6OnN0cmluZyBiZXN0ID0gIi4uLiI7CiAgICAgICAgICAgIHdoaWxlIChsbyA8PSBoaSkgewogICAgICAgICAgICAgICAgY29uc3Qgc2l6ZV90IG1pZCA9IGxvICsgKGhpIC0gbG8pIC8gMjsKICAgICAgICAgICAgICAgIHN0ZDo6c3RyaW5nIGNhbmRpZGF0ZSA9IHRleHQuc3Vic3RyKDAsIG1pZCkgKyAiLi4uIjsKICAgICAgICAgICAgICAgIGlmICh3aWR0aChjYW5kaWRhdGUpIDw9IG1heFcgLyBtaW5TKSB7IGJlc3QgPSBjYW5kaWRhdGU7IGxvID0gbWlkICsgMTsgfQogICAgICAgICAgICAgICAgZWxzZSB7IGlmIChtaWQgPT0gMCkgYnJlYWs7IGhpID0gbWlkIC0gMTsgfQogICAgICAgICAgICB9CiAgICAgICAgICAgIHJldHVybiBiZXN0OwogICAgICAgIH07CiAgICAgICAgY29uc3RleHByIGZsb2F0IGtNYXggPSAyNDAuZiwga01pbiA9IDAuNWY7CiAgICAgICAgLy8gc2hvcnQgdGV4dCB1bnRvdWNoZWQKICAgICAgICBhc3NlcnQoZml0KCJTaG9ydCIsIDAuOGYsIGtNYXgsIGtNaW4pID09ICJTaG9ydCIpOwogICAgICAgIC8vIDI4MHB4IGZpdHMgdGhlIGZ1bGwgYnVkZ2V0IGF0IGJhc2Ugc2NhbGUsIHNvIGl0IHN0YXlzIHdob2xlCiAgICAgICAgc3RkOjpzdHJpbmcgbWVkaXVtKDI4LCAneCcpOwogICAgICAgIGFzc2VydChmaXQobWVkaXVtLCAwLjhmLCBrTWF4LCBrTWluKSA9PSBtZWRpdW0pOwogICAgICAgIC8vIDQwMHB4IG5lZWRzIGEgc2hyaW5rIHRvIDAuNiwgYnV0IHN0aWxsIGZpdHMgd2hvbGUgYWJvdmUgbWluU2NhbGUKICAgICAgICBzdGQ6OnN0cmluZyBzaHJpbmsoNDAsICd5Jyk7CiAgICAgICAgYXNzZXJ0KGZpdChzaHJpbmssIDAuOGYsIGtNYXgsIGtNaW4pID09IHNocmluayk7CiAgICAgICAgLy8gNjAwcHggY2Fubm90IGZpdCBldmVuIGF0IG1pblNjYWxlIC0+IGN1dCB3aXRoIGVsbGlwc2lzCiAgICAgICAgc3RkOjpzdHJpbmcgaHVnZSg2MCwgJ3onKTsKICAgICAgICBzdGQ6OnN0cmluZyBjdXQgPSBmaXQoaHVnZSwgMC44Ziwga01heCwga01pbik7CiAgICAgICAgYXNzZXJ0KGN1dC5zaXplKCkgPCBodWdlLnNpemUoKSk7CiAgICAgICAgYXNzZXJ0KGN1dC5zaXplKCkgPj0gMyAmJiBjdXQuc3Vic3RyKGN1dC5zaXplKCkgLSAzKSA9PSAiLi4uIik7CiAgICAgICAgYXNzZXJ0KHdpZHRoKGN1dCkgPD0ga01heCAvIGtNaW4gKyAxZS0zZik7IC8vIGZpdHMgdGhlIGJ1ZGdldCBhdCBtaW5TY2FsZQogICAgICAgIHN0ZDo6Y291dCA8PCAiICBsYWJlbCBmaXQ6IDYwLWNoYXIgbmFtZSAtPiAiIDw8IGN1dC5zaXplKCkgPDwgIiBjaGFycyB3aXRoIGVsbGlwc2lzXG4iOwogICAgfQoKICAgIHN0ZDo6Y291dCA8PCAiQUxMIExPR0lDIFRFU1RTIFBBU1NFRCIgPDwgc3RkOjplbmRsOwogICAgcmV0dXJuIDA7Cn0K",
 "about.md": "PiBUaGlzIG1vZCB3YXMgcG9ydGVkIHRvIGdlb2RlIHVzaW5nIEFJLiBJZiB5b3UgZG9uJ3QgbGlrZSB1c2luZyBBSSBwb3J0ZWQgbW9kcywgZmVlbCBmcmVlIHRvIG5vdCB1c2UgaXQuDQoNCiMgU29uZyBTZWFyY2gNCg0KQW4gaW4tZ2FtZSBzZWFyY2ggZGlhbG9nIGZvciB5b3VyIGRvd25sb2FkZWQgc29uZ3MsIHdpdGggZnV6enkgbWF0Y2hpbmcgYW5kDQpKdWtlYm94IE5PTkcgc3VwcG9ydC4NCg0KIVtTb25nIFNlYXJjaCBvcGVuIGluIHRoZSBlZGl0b3JdKGZyYW1lOm1hdGNvb2wuc29uZy1zZWFyY2gvc2NyZWVuc2hvdC5wbmc/d2lkdGg9MzAwKQ0KDQpbRnVsbC1zaXplIHNjcmVlbnNob3Qgb24gR2l0SHViXShodHRwczovL2dpdGh1Yi5jb20vbm90bG91c3lib29rL3Nvbmctc2VhcmNoLWdlb2RlL2Jsb2IvbWFpbi9zY3JlZW5zaG90LnBuZykNCg0KVGhlIHNlYXJjaCBkaWFsb2cgcmVwbGFjZXMgdGhlIGRlZmF1bHQgc29uZyBicm93c2VyIGluIHRoZSBlZGl0b3IncyBzb25nIG1lbnUuIEFzIHlvdQ0KdHlwZSwgc29uZ3MgYXJlIHJhbmtlZCBieSBob3cgd2VsbCB0aGV5IG1hdGNoIOKAlCBtYXRjaGluZyBhY3Jvc3MgYm90aCB0aGUgc29uZyBuYW1lIGFuZA0KdGhlIGFydGlzdCDigJQgc28gc2hvcnQsIGFwcHJveGltYXRlIHF1ZXJpZXMgc3RpbGwgZmluZCB0aGUgcmlnaHQgdHJhY2suDQoNCiMjIEZlYXR1cmVzDQoNCi0gKipGdXp6eSBzZWFyY2gqKiBhY3Jvc3Mgc29uZyBuYW1lIGFuZCBhcnRpc3QNCi0gKipMb25nIHRpdGxlcyB0YW1lZCoqOiBuYW1lcyBzaHJpbmsgZHluYW1pY2FsbHkgdG8gZml0IHRoZWlyIHJvdywgc28gdGhlIGZ1bGwNCiAgdGl0bGUgc3RheXMgdmlzaWJsZTsgb25seSBhYnN1cmRseSBsb25nIG5hbWVzIGZhbGwgYmFjayB0byBhIGNsZWFuICIuLi4iIGN1dA0KLSAqKlNvbmcgcHJldmlldyoqIHdpdGhvdXQgbGVhdmluZyB0aGUgZGlhbG9nLCB3aXRoIHRoZSBwbGF5IGJ1dHRvbiB1cGRhdGluZyBwZXIgcm93DQotICoqRGVsZXRlIHNvbmdzKiogc3RyYWlnaHQgZnJvbSBhIHJlc3VsdCByb3cgKHdpdGggYSBjb25maXJtYXRpb24pDQotICoqSnVrZWJveCBpbnRlZ3JhdGlvbioqOiBlYWNoIHJvdyBjYXJyaWVzIHRoZSBzYW1lIHBpbiBkaXNjIEp1a2Vib3ggZHJhd3Mgb24gaXRzDQogIG93biBzb25nIHdpZGdldHMsIHBpbm5lZCBvbiB0aGUgcm93J3MgdG9wLWxlZnQgY29ybmVyOyBjbGlja2luZyBpdCBvcGVucyBKdWtlYm94J3MNCiAgbm9uZyBwaWNrZXIgZm9yIHRoYXQgc29uZy4gVGhpcyBpcyBlbnRpcmVseSBvcHRpb25hbCDigJQgd2l0aG91dCBKdWtlYm94IGluc3RhbGxlZA0KICBub3RoaW5nIGNoYW5nZXMgYW5kIHRoZSBtb2Qgd29ya3MgdGhlIHNhbWUuDQotICoqU2V0dGluZ3MqKiBmb3IgZXZlcnkgcGFydCBvZiB0aGUgYWJvdmUsIGluY2x1ZGluZyBmdWxseSBkaXNhYmxpbmcgdGhlIG1vZA0KDQojIyBDcmVkaXRzDQoNCi0gKiptYXRjb29sKiog4oCUIHRoZSBvcmlnaW5hbCBtb2QsIGluIFtzbWFsbC1nZC1tb2RzXShodHRwczovL2dpdGh1Yi5jb20vbWF0Y29vbC9zbWFsbC1nZC1tb2RzKQ0KLSAqKkZsZXltIC8gRmxhZnkqKiDigJQgW0p1a2Vib3hdKGh0dHBzOi8vZ2l0aHViLmNvbS9GbGVleW0vanVrZWJveCkgYW5kIEF1dG9Ob25nLCBmb3IgdGhlDQogIE5PTkcgc3VwcG9ydA0KDQojIyBQb3J0aW5nIG5vdGVzDQoNClRoaXMgaXMgYSBwb3J0IG9mIHRoZSBvcmlnaW5hbCBXaW5kb3dzLW9ubHkgTWluSG9vay1iYXNlZCBETEwgdG8gYSBwcm9wZXIgR2VvZGUgbW9kLg0KVGhlIG9yaWdpbmFsIGhvb2tlZCBgQ3VzdG9tU29uZ0xheWVyOjpvblNvbmdCcm93c2VyYCBhdCBhIGhhcmRjb2RlZCBvZmZzZXQ7IHRoZSBwb3J0DQp1c2VzIEdlb2RlJ3MgYCRtb2RpZnlgIGhvb2sgaW5zdGVhZCwgd2hpY2ggaXMgd2hhdCBtYWtlcyBpdCB3b3JrIGFjcm9zcyBHRCB2ZXJzaW9ucw0KYW5kIHBsYXRmb3Jtcy4NCg0KVGhlIE5PTkcgaW50ZWdyYXRpb24gZGVzZXJ2ZXMgYSBub3RlOiBKdWtlYm94IGRvZXMgbm90IGV4cG9zZSBhIHF1ZXJ5IEFQSSBmb3Igb3RoZXINCm1vZHMsIGFuZCAodmVyaWZpZWQgYWdhaW5zdCBpdHMgMy44LjAgc291cmNlKSBpdHMgcGluIGRpc2MgaXMgZHJhd24gb24gKmV2ZXJ5KiBzb25nDQp3aWRnZXQgdW5jb25kaXRpb25hbGx5IOKAlCBpdCBpcyB0aGUgYnV0dG9uIHRoYXQgb3BlbnMgaXRzIG5vbmcgcGlja2VyLCBub3QgYSAidGhpcyBzb25nDQpoYXMgYSBub25nIiBiYWRnZS4gU28gdGhlIHBvcnQgbWlycm9ycyB0aGF0IGxvb2s6IGl0IGJ1bmRsZXMgYSBjb3B5IG9mIHRoZSBkaXNjIGFydHdvcmsNCihNSVQpLCBwaW5zIGl0IG9uIGVhY2ggcm93J3MgdG9wLWxlZnQgY29ybmVyLCBhbmQg4oCUIHRvIG1ha2UgaXQgY2xpY2thYmxlIOKAlCBidWlsZHMgYQ0KdGhyb3dhd2F5IGBDdXN0b21Tb25nV2lkZ2V0YCBmb3IgdGhlIHJvdydzIHNvbmcgYW5kIGFjdGl2YXRlcyB0aGUgZGlzYyBidXR0b24gSnVrZWJveA0KYXR0YWNoZXMgdG8gaXQuIFRoZSBwaWNrZXIga2VlcHMgdGhhdCB3aWRnZXQgYWxpdmUgd2hpbGUgaXQgaXMgb3Blbiwgc28gdGhlcmUgYXJlIG5vDQpob29rcyBpbnRvIEp1a2Vib3ggYW5kIG5vIHN0b3JhZ2UgZ3Vlc3NpbmcuDQoNCkNvbnRyaWJ1dGlvbnMgYW5kIGJ1ZyByZXBvcnRzIGFyZSB3ZWxjb21lIOKAlCBpZiBzb21ldGhpbmcgbG9va3Mgd3JvbmcsIGl0J3MgbGlrZWx5IGluDQp0aGlzIHBvcnQgcmF0aGVyIHRoYW4gdGhlIG9yaWdpbmFsLg0K",
 "resources/nong_disc.png": "iVBORw0KGgoAAAANSUhEUgAAAJYAAACXCAYAAAD3XaJHAAAABGdBTUEAALGPC/xhBQAAACBjSFJNAAB6JgAAgIQAAPoAAACA6AAAdTAAAOpgAAA6mAAAF3CculE8AAAABmJLR0QA/wD/AP+gvaeTAAAACXBIWXMAABFtAAARbQFSEUH3AAAjTklEQVR42u2deXiTVfbHb7qkG11ooSstpZStTdK9peleSksXShegQKELBWoBEQFRBEVGHVAYRWUURUAcBYEpgorsMCyCwBRZB3H4KZUOiIhFoEjp8v39QUKT9N43b9q0eQN5n+c8D0/om9zc95N7zj3n3HMIAPIYijeAJAClABYAWAPgKwBHAPwA4AaA3wHcRct1W/HarwDOAzgEYAuAFQDmAigEEAPA5TGdUzV5HL5kL8VDfwvAPgUcHX3VANgGYBGAYQC6m8AyfnEHUALgMwBXIJzrewDvA8gF4GACyzikD4B5AE4AaIbwr/sA9gKYDMDNBJawpBuAGQqYjPlqBLAbQBEAGxNYhpNEAOsA3MOjd9UCeAeAxARW54glgBEAvsXjcx0CMBSAyASW/kUMoALAz/p+aj/++CN27NiB999/H88//zzGjRuHjIwMREVFwd/fH76+vujatetDcXZ2hp+fH/r27Yvo6GgMHToUJSUlmD9/PlavXo19+/bh6tWrHQHYCQDZxgaYUAdmDmA8gJ/08WQuXbqETz/9FFOnTkVMTAwcHBxACOkQcXV1RUpKCmbPno3Nmzfj2rVr+gLsGIA0E1htlyQAJ9tlqNTWYuPGjSgrK0OPHj06DCK+0r9/f0yfPh3bt2/Hn3/+2V7AdhuDDSakwfgA2NQemNasWYOsrCyIxWKDw8QSGxsbZGVlYc2aNbhz50573BVLAdibwGKLCMAkALd0nd3m5mbs3r0bBQUFsLKyEixMLHF0dERFRQVOnGizx+SKwuFqAovi2Dys62zevn0bb7/9Nvz9/dv9cM3NzeHk5AR3d3d4eXnB3d0dLi4ucHZ2ho2NDWxtbWFraws7Ozs4OzujW7du8PDwgKenJ9zd3eHg4ACRSNTucURERGDdunVoaGhoC2BrAXQ1gfVAJigCu7yv3377DXPmzEHXrl3b9PC6dOkCDw+PNt/PB1IleNbW1m16j549e2Lp0qW4e/eurnD9rLBPH1uwHAFU6mo/vfDCC7C3t9fZnvHw8DComrSzs4OHhwcsLS11us/T0xN///vfUV9fr8tUNQH4q2JX/ViBFQjgAt9ZamhowDvvvAMXFxedHqSrq6te1FNHrGgeHh46bS569+6NTZs2tWXn2P1xAWuELqpvz549CAgI4DX5IpEI7u7uMDc3NxrD3draGq6urrz/Pjk5GefOndMFrmoAYY86WE8plmmt1++//47x48fzWnFsbGx0Ws2EKq6urrCwsND6d1ZWVliwYIEu6vGOIh/skQPLQpFhyevauXMnPDw8eKk7Xe0tYxAXFxdealIikeDs2bO6ZE9Me5TAsgKwma8tNXfuXJiZmWldoToyHCMU6datm9YVzNbWFqtWrdJFNb78KIBlC2A7n2/7yy+/ICUlhXMSzczM4Ozs/MgDRdsZavubcePG4fZt3qbrm50VzO4oqPbx+Zbbtm1D9+7dtaoHIe7wOkvEYjEcHR21xiJPnz7NF653OwOujlB/vFaqN954gxMYS0vLx0Lt6WJ/cc2Xra0tvvzyS75w/c2YwLLkY1M1Nzdj/vz5nJNob2//WK9SXKuXra0tp59sxQree6UFxgKW1m/U2NiIiRMnck6eaZXSLlwhKZFIhPnz5/Nz0zc1PSF0sJ7T9iXu3buH/Px8TtXH9Ws0SWu4uHbR06ZNQ1NTkzbt0djY2JguVLBGaDtydffuXcTHxzMnwcnJCdnZ2YiKijJBo4NYWVlxxiDHjx+P5uZmbVqk7s6dO0FCAytQW5imsbEReXl5nD6b/Px8lJaWorS0FH379jVBo4OYmZnBxsaG+f+zZ8/WqhLv3r17+ebNm12FApY9gP9oM9TLysqYX9rDwwMjR458CFVpaSmKioq0uiBMQt/wsP5vyZIlWuG6fv36gQ0bNpgLASytqS+zZ89mfllfX1+MGTNGDSqlFBQUmOwtPRr1IpEIq1ev1grXuXPnlhoarAl8/FRcqSBFRUVUqJSSmZmpNcRjEnq6M2tztG3bNq3G/MaNG/MMBVZvbfnpe/bsYULh6emJwsJCTqiUEh0dbYKljZmyrNfPnz+vLe37anFxce/OBstMW4761atX4e7uTv1i3bt3R0FBAS+oTMZ8+4RlSkilUq1pz6dPn95KCLHrTLCmaXG4ITk5men4zMvL0wmq0tJSFBcXw83NzQRLG3aLLFdERUWFNpXY/Nprr00nhFh0Blg+2lTg3LlzmaGI7OxsnaFSyqhRo0zGfBvPMLL+77PPPuOE6+bNm5d79uwZ3Rlgfc41kL179zJTg+Pj49sMlVKysrKMKvVY6DvFLl264Pvvv9eWHv4pIaRHR4KVrC2dmJX5OWDAgHZDpZS4uDgTLG0QJycn6utyuZzTM9/Q0PBnampqMSHEtiPAMgdwigsslhPU1dWV9w6Qr/Tv398Eix7VorYs1P/85z/7CCGxHQHWeG0qkJbiYmlp2S67isuYZ+06TcJ9Koi1U79x4wanIV9RUTGDENJTn2BZAviRK1c9MDCQOuDY2Fi9Q6VqzNvZ2ZmA0VFYobLy8nLOVeunn346SgjJIoRY6Qsszn3pO++8Qx2ot7c3iouLOwys0tJSZGdnm4z5NmZE0FwTR44c0Raee5YQEq4PsCy5KundvHmTeqbPzMysQ1QgTbhScUyim0qMiIjgNOSrq6v/TQjJI4Q4txescVwEv/jii9QBhoWFdQpUShkwYIAJGB2FdeKpspIzr6C5rKzsKUJIYnvBOsFV9YWWQmxra6tzyEYfxjyfQ64mUdcqtA1XYGAgZ9bp2bNndytWrR5tBSuxLR52uVzeqVApZcyYMY/kyeiOFFaYbMOGDVwhu/tyubyUEJJKCBG1BSymv7+uro5qWzk4OGDs2LEGAau0tBTDhg3jVf/AJOorl+Zr4eHhnEb89u3b1yhWLR9dweoGjuL8y5Ytow4yJibGYFCZjPm2CcsfuH//fiZYf/zxx//Mzc3zta1atBdncKUa09JY7Ozs9O5hb6uw/Gom4S+5ubmcq9ZLL700T7FqeeoC1ndcXnbaQKKiogQBlcmY18+qZWFhgStXrnDla+1UgJXAF6w+XKSOHj2aqqdHjBghGLD0bcxbW5hBbP7opkiznMwLFy5kclBfX/+Ho6PjCC6/luYLL3A5RGle24CAAEFBpZScnBydcuY9ulghxdcFU8J8sDCxL1ZlSrAxN/ihbMgJxor0QCyI88fE4B6I7dEVTtaWjwRctOyHfv36carDxYsX/0UBVhgfsJi+q9WrV1MHlZKSIjioSkpKeKU0O1hZILuPK/42qL8aRHxlQ24wFsT5I6mns1Gvaqycre++Y1pFOHPmjFIdDiOEWHKB5c51ojkjI4PqYujomKCuMm7cOHh7e3OfZLGyQInUC59ky9oEFE0+zJAgu48rLI0UMJrD9Lnn2JUT6urqflOAlUcI8eMCq4Rji0ktYdjZ4Rs+hjtX7xyRiCDNrxs+ypIyAfliVBB2Fsuwb6IUByukODRZioOTJTjwhAR7J8iwvViGzSODmPcvSx2AEDfjK2xCK7Tbpw+nyQ1FiIdqxPNyilZWVlIHk5aWJiiwuDpV2Ist8LzcjwrD1jFBOPCEFFUzA/HdM/zk+IxA7JsoxZYCOmTlId6wNDOeUkysKj///e9/+ThL8wghNiywalhvQCs9ZGtrKyg1GBkZyZw0HwdrLB8S2Orhfz1Ohm+nSXjDxJJvpkjxxajWgL2a0Af2YuOOBixbtowJ1qVLl46rgNWHBlYvriWvZ8+erT5QJpMJBqqhQ4cyd4B9ne1aqb7PRwThmylSKiQ7KyR4d6QUs9OCUBgbimEDw5ATHYaShFC8kBmEVWOk2D+19X0nngnE/klSVOarw7U0pT9cbCyNNuth6NChXG6HWwovfCt1qPxHIevmy5cvUweRkJAgCKiKioqYy7iPg3UrqL4eG4R/a6i8I9MDsTBXhqTQCETKYiEPSURc+CAkRKYiMSoNiVFpSIgcjLjwQYgOSUCENAZDI8LxXoEMxzXe6+h0Cb4YpQ7Xmyn90UVsbpSBaWdnZ848reLi4mkKsHIJIWJNsN5i3bhu3TrqIFRLDxlSQkJCmDs/TfW3s1imBsG/ZwTilWFBGCgbCHloIhKj0pAiz+QlCZGpiA5JQFxwJFaMVl/9qmYG4qsx6qrxL/F9YC7w8pcsZylXR4y1a9e+raIOe2iC9S/WjU899RTV0BMCVAUFBdTJEIkI5sp7c0K1q0KCtPAIRIckIHngEN5AaUpiVBoGBsVhbFwIDj+lohpntYarMNA4Q00ffPABE6xjx459oQJWmCZYzGMatIwBiUQi6IBzml+3VupPFaqNpRJEywYiLjylzUBpSmxYMpJCI7F3skRt5dqi4prYkBOMvs52RucsnTx5MhOsmpqaUypgpamC1YMrm4Hm7jdUQp+qjB49mpp/5WhloWZXfT5C3abaWCpBhFSOxMhUvUGllLjwFMQFR2HfFBWba5oE/8xrgWtxcj+YCVglenl5tXotLi6Oq7ZsrQpYD90Oyube9M6KP//M7ERlaLAiIiKoYyuReqmtVqq7v10VEkQFDewQqFThGhQagaNPt8C1Z4L6BiLeu6tgwaIlcTo5OXEa8P369StUActLCVYps+nd7t2CNdxpW2MHKwu1MM3X42RqhvqQiAi9qj+WxIQmYUJisJq9peqtfzOlP0Qi4wntEEJw7do1JlhPPPHE0ypgSZVg/YV1w4oVK6hpMoZ2jObm5lK/fHYfV7WVQdX5+cqwIESHJHQ4VEqJCorFZ8Utn7+/XH3VknY3rhz9o0ePgiOreJEKWLFKsD5i3TBv3jzqaVqhetlVsxS2jmkx2A9PD8RA2cB27f50lYTIwUiLiMCJWS2r1qbhLWBNCfMxKrDWr1/PBOvLL79cqQJWhhKsrawbSkpKWnuy+/Y1OFg+Pj7UfCrVFeHAEy221cJcGeShiZ0GVcuqFYf1pS2r1q7SFjX9UZZUsEY8rRYZV+XlgwcPbtQw4K0IAOYal5WV1eoDgoODDQ4W7TRviq+LGliqAeWk0AidnJ/6NOTLVGytw1PVkwd7dxVmMblu3bq1em3OnDlMsE6ePLldAywXAuD/WDcMHDiw1QdERkYaPO2YNhlTwnzUUl9UY3+RsthOhypFnolB0RmIkkU/dHecmBWIf6rEEjN6C7OePe3MwMSJE5lgXbx48RsNsHwIgF85tpGC82FlZmbSc7QT+1K97O+OlEIekmgQsJTqcOvEFnWomgUxMbiH0fiy8vPzwRFP/k4DrP4EwB+sG2i2jKHPDyYlJVEnY3Vmy65r38QW++qZIUGICx9kMLDkIYlqscTtRS1gvRjbWzAuBrFYDGtra9jb21OzWTIyMphgXbly5awGWKEEwJ+sG2hHg+Li4gwKVmxsbOuJIQQbclQM94qWBzkmNhQJHegQ5RPqeXWYjGrAv5bUj3lCWdmb0N7eHk5OTnBxcYGbmxs8PT3h7e0NX19f+Pv7o1+/fggICIBUKkVISAjCw8MRFRUFuVyO+Ph4JCUlISUlBWlpacjMzER2djZyc3NRUFCAwsJCaiMHmvM5JSWFCda1a9e+1wArmgBgVoGgOSENDRbN7rMyN1Mzig9NbgErJzrMIIZ7iwE/CHMyWmy+PWUtYL09mF4pJz093aBzTOvCxhXWuXHjxv9pgJVAADSybqC59w0NFu1LizXBUlmx8gQA1guZqmBJ1ZIAaWBlZGQIbo7j4+O5Gjxd1ABrMAHAbFNA2x2oloAsKipCYWEhCgoKMHz4cOTm5iI7OxuZmZlIS0tDSkoKkpKSEB8fD7lcjqioKISEhEAqlSIgIAD9+vWDv78/fH194e3tDU9PT7i5ucHFxQVOTk6wt7eHra0txGKx1jOCn+UEqfiwWozlovhQJEQONqgqXJRDV4ULE+lH1IYNG2ZQsMLDw1uNafDgwUywfvnll/MaYGUQADdZN/j6+urUNtaQsiK9Jalv74SWBzkvQ4ZYAxrv0SEJ+KiwZQXdNq4FrOflftTvMnz4cIOCFRYW1mpMWVlZXMb7GQ2whhIAv7BuCAgI4OU8E4IsiPN/+MC2q7gbVo2RdmqMUFMiZbHYpbKCqp7qKZF6UXdo2jqjdbRIpdJW4xoxYgQ4smCqNMAaRgAw2xPQCvYLtRT2xOAeDx/Y5pEtNs3+qYGIkMYYBKrkgemQB0U9jBdWzZRgY16LLZji60JtaCnEY3RcyX4XLlw4oAFWLgFwiHVDTk4OL6+sECTWu6uaAX98RktIZ2hEuEFcDrFhgzAjNeThOA5NVg/peHSxolacNjRYtMXjhReYZT1QVVX1FQ2sLbqcJxRqJy4na0tsUHlo+ya12DXvFcgMog4jZbHYOklCdY4uH0JPqw4PDzc4WLQuFm+9xTxvg/37939GU4UrWDcsWLBAp97DQrKzthS0qMPjMwMRFxzZqW6HuPAUFMSEquW/q545LKbYV8pzfIZO+aaN6/PP2T26Kisr36cZ7/NYN3z88ce8MwyFIMk9XTTSkltWiw9GSzEwKK7Tgs+R0hhsK5dQ/Vcbc4Ph62hDrcFeUlJiULBoGS3aKs8sWbLkZZq7gXlY9cCBA0aVjCY2N8OHGS12zJejg3BC5XROYWwoYsKSO8XFMH9oy4pZNUOCTSqr1Utx9BoTQqg1xorF1tbWMsEqKip6kuYgjeHwqOp0sFEIMkwjPflf5VK1TNKk0MgOzXuPCU1CQUwoqma1AL2jWMYrLbmzOnpwSUxMDHXDxtVYwM3NrYAW0nHhuou2C6SFeoS0ai1LDXj4ECvzg3F0uopKmixBbFBUh8AVE5qErMhwtRM6mjvB56L9mGWEhHBWUyKRtBpbamoqV3PyaxpQPQxCEwD/Y92YlpZmNL4spYS6O6g9zC0FQaia2QLX3imBGBQagZjQJL3ZVNEhCSiICVWD6uh0dRX4SbYMrrZi6phTU1MFARbtHOnMmTO5OoMdo4AVqgTra9aNc+bM4dVBSmhSFtRDDa6vxgThxKwWuI4+HYiypBBEBcW2K5YYF56CSGkM5mfJ1NTf8acl2DxSvThIUk9nQa9W+fn51PF9+umnuuS7P0z0IwAWcZzAMMp6A5ZmIvxVJav0gTEva1Vc7bMSCdIiIhAVFPfgsGl0Bi+PemzYIETKYjEyJhQ7ytVrbB2b3hqqilAfZpJdVlaWIMBKSEigjrG6upoJ1sqVK5dQwPJWgjWMq98zLbPAGBpR2ost8NbgARpqMRhHn1IH4cSsQHxWLMH4hBBEyaIRFRQHeUgiYsOSERc+CHHhgxAblozokIQHZY6CojAjNQRfTWpdtO3QZHX1pww2mzOq++mzZ3ZHBJ+9vLw4y0Wmp6dPooDlrNrmhHmGmmbQGUv7XBcbSyxNUa+K/M+8YOydIH0Yw1MrbTQzEF9NlOCD0VK8ki3D3MwgzM2UYeEwGVYXSrHziUDqfVUzJK12f0qorBgFb52cnDBu3DjBVJqm2VdjxoxhQnXnzp1fKVDlEULEqqUiL7DeYNasWdTuBcaiFu3FFng5vk+rh755ZBD2l0vUfF26StXMB7UZKoe3LhVZEerDXKksLCyQm5srqKqItHF+/PHHXMHn/RSoMjRrkL7PeoNdu3YZnT+r1VhFIoyVeKrFE1sq0gRjV4kMR56kr2I0mA5NlmB7kaxVaUjl7o9lqCvtKiEUVlEVuVxOHecvvzCzqrBx48b3KGDFaIKVy1GqhlqO0Ri7yfdztsPi5H7Mctr/zH/gsd9eFITd42XYUybFnjIZdpVKsW2cDFtGBamlvmjKc9F+cLMTc2didGAj9rYKzV8ZERHBaV9lZmbS7CupJlgOAO6z3qSwsLC1M1IsNs7+MSIREnycW9le7ZEFcf5aC32IRCJBtN+jNW6njff1119nQlVbW1vNsK+8aC1P9rLeaMuWLTo1rzaObgwEMld7TA3zwRqOpgIseX9IIEqkXujlZMOrvqcQ28OwDk+IRCL89NNPTLC+/fbbzQywrGlgTeZSh7R89+7duxstWGpn+UQi+He1RUbv7pgU7I35sf54LakvlqUG4O3BA/BaUl88L/dDqcwLKb4u8LLX7QdlZWWFkSNHCg6qoqIidOnSpdV4o6OjOdXgzJkzZ1OgSmU1aXLjOg725JNPmnr8tbNanlDcC0pJTk6mH05ZsYIzPqhS310tlMPVCHM3RwNE3n1YTEKX3r17Cwos2nH6Ll264NatW0ywjhw5sonLvmKBVcS1BEZHRxu120EIIpSOtEOGDKEfTOGoLKPRnEk9z12lvRwNLBsAzKyuDRs28C6KahJ29/ghQ4YYHCxaT0eRSMTZMIBSpyGPq+WJpjA78zQ2NqJ379ZVUiwtLU3Q6GjMG/Jganp6OjPnnutat27dOwyw/PmAJeGKHS5btsy0aumpKZKhjHna4kAIwYEDB7iaMt329fUdzVCDNnzAIgC2cXwA9fi9Lj2YTfJA/Pz8BJPXzlWfgSP3Ko8QEq+tJ7SqpHB9yPLly3nXrzQJt0RERHQaVGPHjmXW3zhy5AjzeTc1Nd1PTEwsY4DlqwtYnM3H79+/z2zobayhHkNW1Bs8eHCngBUaGkodQ25uLudqdebMmV0MqLIJIRa6gjWM68NY2aVCPtT6OBvzOTk5VNeQra0tZ/imqampMTc39wkGWKE0drSBJQJwjAuu1NRUkyGvx85bHWXMl5SUwM+PfkJo4cKFnKvVqVOntjOgyiOEdG0LWARAGteHXrhwgRmINrkgdJeePXt2CFg0x7YyNbq+vh4c7qX6tLS0iXyNdl3A4jzFAwCvvvrqI5f5YEgJCwvTu8+KFR3Zs2ePtp3gBo7VyqO9YA3gytW6f/8+goOD6S10HR1NsLTBmNdXis2oUaOoueza8tkBoK6u7gbDb/XgGD0hovaCRQAs5RrEDz/8wDTaTXC14US3WNzu9n0lJSXMnbuPjw9u3LjBCdY//vGPpRyrVQ8uXnQByx7Az1wDoVWnUf4CTWpRd3F0dMTYsWP1WrpcCe2xY5x7MtTU1JxipMbkEUJSuFYrXcEiANKh5SotLWWeSjFlQeguPj4+bYIqMTGR+Z5Ll3IqHzQ2Nt4bPnz4ZI7VylMbK7qCRQB8yjWoP//8E0FBQSbHqR4lJCREJ6gGDRrEDK9lZWVxtuEFgG3btn3EAVUCH07aApZTU1PTZa6BnTt3jpruaiwnqI3ZmE9PT2f+gHv16oXff/+dE6qrV6+es7GxGc6AKpcQ4tRRYCkblDdpoZ7pxzIZ822oRWFpqfWAa3Z2NvOH6+LigvPnz3NCVV9ff5vDZ8X0susTLHLv3r3Xtdlba9euZS7JJrjaZswXFhZSocrIyGBCZWtri2+++Ubb42p+7733XuOAKpMQIu5wsACY19XVHdI2WlbuFiGEWp3XJNzi7e3dqk7p4MGDmerP0tISX3/9tbbHxHWc62EFGV34aA9YBIDz3bt3r2gb9LPPPssZfDWmOhBCENX2yQkJCUytIBKJsGrVKq1QXbly5UyXLl1GcEAVoysb7QWLXLp0Sd7Y2Pgn5xrb3Izy8nLmRFlYWJhWLx0lMTGRetBUFSqu2uyqR7kGDhxYwgHVUM3s0E4BCwA5fvx4UXNzc5O2L7Fo0SLuJgCM0INJiE4HhS0sLPDhhx9qhaqhoaFuwoQJ07WoQK+2MKEXsACQY8eOLQaPa9myZZwpzLTiIyZRT/9muXKUhvrWrVu1PoempqaGl19++QUtUIW0lQe9gUUIMa+qqvqED1ybNm3iDPGIxWKjqHNqiF0h14/S2dkZhw5p3U+hubm5aeXKlX/TAtUgQoi5wcFSwGV34sSJLXzg2rVrl1aXgyl/nv9c9OrVS6ufSgFV89q1a9/WAlUWIcS2XSzoEywAxM7Ozv3UqVPb+MB16dIlZqDUtHq1mAbaTj9lZ2drzVRQckXpe6MpOYQQ13YvMvoGCwCxtrbudezYMV4r1/379/Hss89q7dHj5OT0WB0vs7Ky0hr+srCwwKJFi7TG/pTqb/369cu0QJVHCOmlF+3VEWAp1KL0wIEDG8Dzqqys5LUrfNQLkJibm/MqDdWrVy+tqS+qhvoHH3ywmAdUgXp7/h0IlogQEl5ZWbm8ubm5kc8E/Pjjj4iPj+e1M3rUADM3N+f9nQoLCzmbJmlohNuvvPLKizygCtXr8+8osFTgilq0aNGChoaGOl5GQHMzPv74Y94NN93d3Y06z8vKygrOzs68/nbAgAFac9Q1nZ/FxcXTeEAVqS1xT1BgKeAyI4TIJ02a9PStW7eu8p2U2tpaTJkyhTc0Dg4ORuVgdXV1ha2tLa+/tbOzw8KFCzlP01DCNGflcnkpD6iiCSFmen/uHQ2WClxREolkXHV19XHocFVVVWndOdK80kI8NOvi4sI83s6SvLw8zpYjtEX/6NGjm7XE/lRXKrMOeeadAZaqzWVubp6/e/fufzQ1NTXynqnmZmzZsoWZmaptN+nm5mYQdSkWi+Hh4cHpKWdJamoqvv32W12AQn19/e3ly5e/zgOoBx269Kz+DAKWClxSQkjezJkzZ9+6det/Ov0Um5tRWVmp8wqm6b329PSEs7OzXtsQW1hYoHv37m0GSbkpyc7O5uU9p2UoaEnS65DdnyDAUgHMlxCS4+fnN+bkyZPbmvk4YjSuQ4cOYfjw4XrLo7exsYGLiwvc3d3h5eUFT09PeHh4wMvLC15eXg//7enpCXd3dzg7O+vtpLe9vT3Ky8t5ec5pBx+2bdv2kVgszucBVK6+/FSCBEsBl5siJSPv+eefn1NbW3sZbbiuXbuGxYsXIyAgwOjy2OVyOT788EPcvn27LV8dV69ePTd27NipPFepLH141AUPljK2qAh25nXv3r1g7969axsaGu6hjdfp06cxb9489O/fX7CZCZGRkVi8eLGuBrnadffu3RuffPLJWxzn/jQlub2xP6MCS5kVoTAk8wghecnJyRNOnz69sy3qUdPZ+t577yEnJ8egTQ58fHwwduxYfPLJJ/j111/b85XQ1NTUUFVV9VW/fv0KeQKVRwgJaU+WgtGCpQKYl1I1EkLyJkyYMP3ixYvfcNVC1eU6f/48Vq5cialTpyIhIUHnbT8f8fDwQGpqKp555hmsXbsWP//8sz6Gjqampsbz58//Kz8/v0IHoLL4HCx95MFSwGVDCIlVnaBZs2Y9W11dXaUvwFSv69ev4/jx49iwYQPeeOMNzJ07F+Xl5RgxYgQyMzORkpKiJtnZ2Rg1ahSmTJmCF198EW+99RY2b96MU6dO4Y8//tD38NDU1HT/7NmzuzmKnjFz1JU9bUxgtV69slQnq6ys7KmzZ8/uaWpquo9H/Lp///7tw4cPV3LU/OTKT+8liGcoRLAUcFkRQsI1J08ul5fu2LFjja4+MGO4fv311+/Xr1+/jKN0EJcbIVSXc3+PLVgqgDkRQhI0J9Pc3Dz/pZdemnfmzJmd9fX1t4wVpjt37lw7cuTIJkYbET6SRAhxFtxzEzpYKoD1UBT7ajW5jo6OI5YsWfLymTNndtbV1f0mdJhqa2svHz16dPPs2bOf1cFlQCsl5CnY52UsYKmEhHwIIalckz5p0qSnd+zYsaa6uvp4fX39bUODVFdXd/3ChQsHKysrlw8dOrS8jSCpVtLz7sg432MHlgZgnjQVSVOZpaWl09atW/fO8ePHv6ipqTl97969PzpStVVXVx8/fPhw5cqVK5ekp6dPaidIDwvJEkI8hA6UUYOlAZkzISRMUcie94MKCAgYW1FRMePdd99dtHXr1lUHDx7cePLkye0XL148XFNTc6qmpubUb7/9dlEp169f/6GmpubU5cuXT164cOHgiRMnvt6/f//6zZs3r3jzzTdfKS4unubm5lagJ4hUi/OHskpem8DqHMAsCSG9+KxiApdcxerkS+v4YALL8I7WPgrIco0EpgRCiH9b6iSYwDIMZGLFjjKMEJImIJjSFGrOS7Uz6SMz7486WBTQrBUPU6oIH2V0AkQZijCLRPHZ1o/8PD9uYHGsai4KV0Z/xUoSrVBPgxXhpaEaG4RsxWuZir9JUNwTqngPb8V7ih/HOf1/TZHCy6UNXEUAAAAldEVYdGRhdGU6Y3JlYXRlADIwMjYtMTAtMThUMDQ6MTI6MDYrMDA6MDCJrQNpAAAAJXRFWHRkYXRlOm1vZGlmeQAyMDI2LTEwLTE4VDA0OjEyOjA2KzAwOjAw+PC71QAAABl0RVh0U29mdHdhcmUAd3d3Lmlua3NjYXBlLm9yZ5vuPBoAAAAASUVORK5CYII=",
 "resources/screenshot.png": "...",
 "logo.png": "...",
 "changelog.md": "IyB2MC4yLjgNCg0KLSB0aGUgcGFja2FnZSBub3cgc2hpcHMgYGFib3V0Lm1kYCwgYGNoYW5nZWxvZy5tZGAgYW5kIGBzdXBwb3J0Lm1kYA0KLSBidWlsZHMgcnVuIG9uIEdpdEh1YiBBY3Rpb25zIGFuZCByZWxlYXNlcyBsaXZlIG9uIEdpdEh1Yg0KLSBidWlsZHMgZm9yIG1hY09TLCBhbmRyb2lkIGFuZCBpb3Mgbm93IC0gdGhlIHJvdy1ob3ZlciBkZWxldGUgYnV0dG9uIHN0YXlzIHdpbmRvd3Mtb25seQ0KLSByZWFkbWUgYW5kIGFib3V0IHBhZ2UgZ290IGEgc2NyZWVuc2hvdA0KLSBtb2QgbGlua3MgcG9pbnQgYXQgdGhpcyByZXBvDQoNCiMgdjAuMi43DQoNCi0gcGlja2luZyBhIG5vbmcgbm93IHJlZnJlc2hlcyB0aGF0IHJvdydzIG5hbWUgYW5kIGFydGlzdCByaWdodCBhd2F5LCBpbnN0ZWFkIG9mIHdhaXRpbmcgZm9yIGEgc2Nyb2xsDQoNCiMgdjAuMi42DQoNCi0gZml4ZWQgdGhlIG5hbWUvYXJ0aXN0IHBsYWNlbWVudCBmb3IgcmVhbCAtIHRoZSBsYWJlbHMgd2VyZSBhIGZ1bGwgcm93IHRvbyBoaWdoIChyb3cgc3BhY2UgaXMgY2VudGVyZWQgb24gdGhlIHJvdydzIG1pZGRsZSwgbm90IGJvdHRvbS1sZWZ0KQ0KLSBuYW1lIHNpdHMganVzdCBhYm92ZSB0aGUgcm93J3MgbWlkZGxlLCBhcnRpc3QganVzdCBiZWxvdyBpdCAtIHNhbWUgZmVlbCBhcyB0aGUgb3JpZ2luYWwgYnJvd3Nlcg0KDQojIHYwLjIuNQ0KDQotIHRpZ2h0ZXIgdGV4dCBwYXNzOiBuYW1lIG1heCBzaXplIDAuNTUsIHRleHQgaHVnZ2luZyB0aGUgcm93J3MgbGVmdCBlZGdlDQoNCiMgdjAuMi40DQoNCi0gbG9uZyB0aXRsZXMgc2hyaW5rIHRvIGZpdCBpbnN0ZWFkIG9mIGdldHRpbmcgY3V0IHNob3J0OyAiLi4uIiBpcyBvbmx5IGZvciB0cnVseSBhYnN1cmQgbmFtZXMgbm93DQotIHRoZSBzZWFyY2ggYmFyIHJlbmRlcnMgYWJvdmUgdGhlIHJvd3MsIHNvIHNjcm9sbGluZyByb3dzIHBhc3MgYmVoaW5kIGl0DQoNCiMgdjAuMi4zDQoNCi0gdGhlIGRpc2Mgb24gZWFjaCByb3cgbm93IG9wZW5zIEp1a2Vib3gncyBub25nIHBpY2tlciBmb3IgdGhhdCBzb25nDQoNCiMgdjAuMi4yDQoNCi0gbGl2ZSBpbi11c2Ugc3RhdGUsIHJlY3Vyc2l2ZSBOT05HIGxvb2t1cCwgc2Nyb2xsaW5nL3NwYWNpbmcgcG9saXNoDQotIGZpbHRlciBidXR0b25zIGRyb3BwZWQgLSBzZWFyY2hpbmcgY292ZXJzIGl0DQoNCiMgdjAuMi4xDQoNCi0gZml4ZWQgZmlsdGVyIGljb25zLCBhIE5PTkcgcGlja3VwIHJhY2UsIGFuZCBzY3JvbGxpbmcgaXMgZWFzZWQgbm93DQoNCiMgdjAuMi4wDQoNCi0gZmlyc3QgR2VvZGUgYnVpbGQ6IGZ1enp5IHNlYXJjaCBhY3Jvc3MgbmFtZSArIGFydGlzdCwgcHJldmlldywgZGVsZXRlLCBzZWxlY3QsIHNldHRpbmdzDQo=",
 "support.md": "IyB0aGFua3MgZm9yIHVzaW5nIHNvbmcgc2VhcmNoIQoKaWYgeW91IGxpa2UgdGhlIG1vZCBhbmQgd2FudCB0byB0aHJvdyBhIGNvaW4gbXkgd2F5OgoKKipba28tZmkuY29tL25vdGxvdXN5XShodHRwczovL2tvLWZpLmNvbS9ub3Rsb3VzeSkqKgoKdGhpcyBpcyBhIHBvcnQgb2YgbWF0Y29vbCdzIFtzbWFsbC1nZC1tb2RzXShodHRwczovL2dpdGh1Yi5jb20vbWF0Y29vbC9zbWFsbC1nZC1tb2RzKQpzb25nLXNlYXJjaCwgc28gbW9zdCBvZiB0aGUgY3JlZGl0IGdvZXMgdG8gaGltIC0gY29uc2lkZXIgc3RhcnJpbmcgaGlzIHJlcG8gaW5zdGVhZCA6KQoKbm8gcHJlc3N1cmUgYXQgYWxsIC0gdXNpbmcgdGhlIG1vZCBhbmQgcmVwb3J0aW5nIGJ1Z3MgaXMgYWxyZWFkeSBwbGVudHkuCg==",
 "LICENSE": "TUlUIExpY2Vuc2UKCkNvcHlyaWdodCAoYykgMjAyNiBub3Rsb3VzeWJvb2sKClRoaXMgcHJvamVjdCBpcyBhIEdlb2RlIHBvcnQgb2YgInNvbmctc2VhcmNoIiBmcm9tIHNtYWxsLWdkLW1vZHMKKGh0dHBzOi8vZ2l0aHViLmNvbS9tYXRjb29sL3NtYWxsLWdkLW1vZHMpLCBDb3B5cmlnaHQgKGMpIDIwMjEgbWF0LCB3aGljaCBpcwphbHNvIGxpY2Vuc2VkIHVuZGVyIHRoZSBNSVQgTGljZW5zZS4gSXRzIG5vdGljZSBpcyBwcmVzZXJ2ZWQgYmVsb3cgYXMKcmVxdWlyZWQuCgpUaGUgYnVuZGxlZCBOT05HIHBpbi1kaXNjIGFydHdvcmsgKHJlc291cmNlcy9ub25nX2Rpc2MucG5nKSBjb21lcyBmcm9tCkp1a2Vib3ggKGh0dHBzOi8vZ2l0aHViLmNvbS9GbGVleW0vanVrZWJveCksIE1JVCBsaWNlbnNlZC4KCi0tLS0KCkNvcHlyaWdodCAoYykgMjAyMSBtYXQgKHNtYWxsLWdkLW1vZHMpCgpQZXJtaXNzaW9uIGlzIGhlcmVieSBncmFudGVkLCBmcmVlIG9mIGNoYXJnZSwgdG8gYW55IHBlcnNvbiBvYnRhaW5pbmcgYSBjb3B5Cm9mIHRoaXMgc29mdHdhcmUgYW5kIGFzc29jaWF0ZWQgZG9jdW1lbnRhdGlvbiBmaWxlcyAodGhlICJTb2Z0d2FyZSIpLCB0byBkZWFsCmluIHRoZSBTb2Z0d2FyZSB3aXRob3V0IHJlc3RyaWN0aW9uLCBpbmNsdWRpbmcgd2l0aG91dCBsaW1pdGF0aW9uIHRoZSByaWdodHMKdG8gdXNlLCBjb3B5LCBtb2RpZnksIG1lcmdlLCBwdWJsaXNoLCBkaXN0cmlidXRlLCBzdWJsaWNlbnNlLCBhbmQvb3Igc2VsbApjb3BpZXMgb2YgdGhlIFNvZnR3YXJlLCBhbmQgdG8gcGVybWl0IHBlcnNvbnMgdG8gd2hvbSB0aGUgU29mdHdhcmUgaXMKZnVybmlzaGVkIHRvIGRvIHNvLCBzdWJqZWN0IHRvIHRoZSBmb2xsb3dpbmcgY29uZGl0aW9uczoKClRoZSBhYm92ZSBjb3B5cmlnaHQgbm90aWNlIGFuZCB0aGlzIHBlcm1pc3Npb24gbm90aWNlIHNoYWxsIGJlIGluY2x1ZGVkIGluIGFsbApjb3BpZXMgb3Igc3Vic3RhbnRpYWwgcG9ydGlvbnMgb2YgdGhlIFNvZnR3YXJlLgoKVEhFIFNPRlRXQVJFIElTIFBST1ZJREVEICJBUyBJUyIsIFdJVEhPVVQgV0FSUkFOVFkgT0YgQU5ZIEtJTkQsIEVYUFJFU1MgT1IKSU1QTElFRCwgSU5DTFVESU5HIEJVVCBOT1QgTElNSVRFRCBUTyBUSEUgV0FSUkFOVElFUyBPRiBNRVJDSEFOVEFCSUxJVFksCkZJVE5FU1MgRk9SIEEgUEFSVElDVUxBUiBQVVJQT1NFIEFORCBOT05JTkZSSU5HRU1FTlQuIElOIE5PIEVWRU5UIFNIQUxMIFRIRQpBVVRIT1JTIE9SIENPUFlSSUdIVCBIT0xERVJTIEJFIExJQUJMRSBGT1IgQU5ZIENMQUlNLCBEQU1BR0VTIE9SIE9USEVSCkxJQUJJTElUWSwgV0hFVEhFUiBJTiBBTiBBQ1RJT04gT0YgQ09OVFJBQ1QsIFRPUlQgT1IgT1RIRVJXSVNFLCBBUklTSU5HIEZST00sCk9VVCBPRiBPUiBJTiBDT05ORUNUSU9OIFdJVEggVEhFIFNPRlRXQVJFIE9SIFRIRSBVU0UgT1IgT1RIRVIgREVBTElOR1MgSU4gVEhFClNPRlRXQVJFLgo=",
}
ROOT = pathlib.Path('/kaggle/working/song-search')
for rel, b64data in FILES.items():
    p = ROOT / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64data))
    print('wrote', p, p.stat().st_size, 'bytes')

# the SDK requires mod.json "geode" to match its own version exactly, and
# compares major.minor with the "v" prefix stripped from both sides, so the
# field must carry no prefix
sdk_ver = pathlib.Path('/tmp/work/sdk/VERSION').read_text().strip().lstrip('v')
mj = ROOT / 'mod.json'
m = json.loads(mj.read_text())
if m.get('geode') != sdk_ver:
    print('mod.json geode', m.get('geode'), '->', sdk_ver)
    m['geode'] = sdk_ver
    mj.write_text(json.dumps(m, indent=2) + chr(10))
print('mod.json geode =', m['geode'])

In [ ]:
%%bash

set -eo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "4. mod source + native logic test"
cd /kaggle/working/song-search || exit 10
find . -type f | sort | tee -a "$LOG"
g++ -std=c++20 -O2 -Wall -I src -o /tmp/work/logic_test tests/logic_test.cpp
/tmp/work/logic_test | tee -a "$LOG"
log "logic test rc=$?"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "5. configure (cross-compile -> Win64)"
cd /kaggle/working/song-search || exit 11
# capture-orig-geode: remember the mod's declared minimum Geode version
# before the Geode packaging step rewrites it to the SDK version.
python3 -c "import json;print(json.load(open('mod.json'))['geode'])" > /tmp/work/orig_mod_geode.txt
echo "source mod.json geode: $(cat /tmp/work/orig_mod_geode.txt)" | tee -a "$LOG"
export PATH=/usr/local/bin:/tmp/work/geode-cli:/tmp/work/xwin:$PATH
export SPLAT_DIR=/tmp/work/splat
export TOOLCHAIN=/tmp/work/toolchain/clang-cl-msvc.cmake
export HOST_ARCH=x86_64
export GEODE_SDK=/tmp/work/sdk
export GEODE_BINDINGS_REPO_PATH=/tmp/work/bindings

rm -rf build
cmake \
  -DCMAKE_TOOLCHAIN_FILE=$TOOLCHAIN \
  -DCMAKE_BUILD_TYPE=Release \
  -DGEODE_TARGET_PLATFORM=Win64 \
  -G Ninja \
  -B build > /tmp/work/configure.log 2>&1
RC=$?
tail -50 /tmp/work/configure.log | tee -a "$LOG"
echo "configure rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 12

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
log "6. build"
cd /kaggle/working/song-search || exit 13
export PATH=/usr/local/bin:/tmp/work/geode-cli:$PATH
export GEODE_SDK=/tmp/work/sdk
time cmake --build build --config Release > /tmp/work/build.log 2>&1
RC=$?
tail -70 /tmp/work/build.log | tee -a "$LOG"
echo "build rc=$RC" | tee -a "$LOG"
[ $RC -eq 0 ] || exit 14
# restore-orig-geode: Geode's packaging rewrites mod.json's `geode` field to
# the SDK version, which makes the mod refuse to load on older loaders.
# Set it back to the version the mod actually targets.
python3 - <<'PY' | tee -a "$LOG"
import json, re, glob, os, shutil, zipfile
want = "5.11.1"
t = '/kaggle/working/song-search/mod.json'
txt = open(t, encoding='utf-8').read()
new = re.sub(r'"geode"\s*:\s*"[^"]+"', '"geode": "%s"' % want, txt, count=1)
assert new != txt or ('"geode": "%s"' % want) in txt, 'tree mod.json geode not found'
json.loads(new)
open(t, 'w', encoding='utf-8').write(new)
geos = glob.glob('/kaggle/working/song-search/build/**/*.geode', recursive=True)
assert geos, 'no .geode in build/'
g = max(geos, key=os.path.getmtime)
zin = zipfile.ZipFile(g)
items = []
for info in zin.infolist():
    d = zin.read(info.filename)
    if info.filename == 'mod.json':
        s = d.decode('utf-8')
        s2 = re.sub(r'"geode"\s*:\s*"[^"]+"', '"geode": "%s"' % want, s, count=1)
        assert s2 != s or ('"geode": "%s"' % want) in s, 'zip mod.json geode not found'
        json.loads(s2)
        d = s2.encode('utf-8')
    items.append((info, d))
zin.close()
if 'logo.png' not in [i.filename for i, _ in items]:
    lp = '/kaggle/working/song-search/logo.png'
    if os.path.exists(lp):
        zi = zipfile.ZipInfo('logo.png')
        zi.compress_type = zipfile.ZIP_DEFLATED
        items.append((zi, open(lp, 'rb').read()))
        print('logo.png appended to package')
tmp = g + '.tmp'
with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zo:
    for info, d in items:
        zo.writestr(info, d)
shutil.move(tmp, g)
m = json.loads(zipfile.ZipFile(g).read('mod.json').decode('utf-8'))
print('restored mod geode:', m['geode'], '| version:', m['version'], '| file:', g)
PY
log "BUILD DONE"
find build -maxdepth 3 -type f \( -name '*.geode' -o -name '*.dll' -o -name '*.lib' \) \
  -exec ls -la {} \; | tee -a "$LOG"

In [ ]:
%%bash

set -uo pipefail
LOG=/kaggle/working/BUILD-LOG.txt
log(){ echo "== $* ==" | tee -a "$LOG"; }
cd /kaggle/working/song-search || exit 15
GEO=$(find build -maxdepth 3 -name '*.geode' 2>/dev/null | head -1)
if [ -z "$GEO" ]; then
  echo "NO .geode PRODUCED" | tee -a "$LOG"
  find build -maxdepth 3 -type f 2>/dev/null | head -30 | tee -a "$LOG"
  exit 16
fi
echo "geode: $GEO ($(stat -c%s "$GEO") bytes)" | tee -a "$LOG"
unzip -l "$GEO" | tee -a "$LOG"

log "7. DLL architecture check"
mkdir -p /tmp/work/geocheck
cd /tmp/work/geocheck
rm -rf ./* 2>/dev/null || true
unzip -o -q "/kaggle/working/song-search/$GEO"
DLL=$(find . -name '*.dll' | head -1)
echo "mod dll: $DLL" | tee -a "$LOG"
python3 - "$DLL" <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import sys, struct
def pe_info(p):
    d = open(p,'rb').read()
    assert d[:2]==b'MZ'
    e = struct.unpack_from('<I', d, 0x3c)[0]
    assert d[e:e+4]==b'PE\x00\x00'
    mach  = struct.unpack_from('<H', d, e+4)[0]
    chars = struct.unpack_from('<H', d, e+22)[0]
    magic = struct.unpack_from('<H', d, e+24)[0]
    return dict(
      machine={0x14c:'i386 (32-bit)',0x8664:'x86_64 (64-bit)',0xaa64:'arm64'}.get(mach,hex(mach)),
      is_dll=bool(chars & 0x2000),
      pe_format='PE32+' if magic==0x20b else 'PE32',
      size=len(d))
i = pe_info(sys.argv[1])
for k,v in i.items(): print('  %s: %s' % (k,v))
ok = i['machine'].startswith('x86_64') and i['is_dll'] and i['pe_format']=='PE32+'
print('  ARCH MATCHES GD 2.2081 (x86_64):', 'YES' if ok else 'NO')
PY

log "8. mod.json sanity"
python3 - <<'PY' | tee -a /kaggle/working/BUILD-LOG.txt
import json
m = json.load(open('/kaggle/working/song-search/mod.json'))
print('id     :', m['id'])
print('version:', m['version'])
print('geode  :', m['geode'])
print('gd.win :', m['gd']['win'])
assert m['gd']['win'] in ('*', '2.2081')
print('mod.json OK')
PY

# --- keep the kernel output small ---
# `kaggle kernels output` downloads everything under /kaggle/working, and the
# build tree holds ~30MB of static libs. Copy the deliverables out and drop
# the build tree so the package can actually be fetched.
cd /kaggle/working/song-search
log "9. pruning build tree / staging output"
if [ -f build/matcool.song-search.geode ]; then
  cp -f build/matcool.song-search.geode /kaggle/working/matcool.song-search.geode
  cp -f build/matcool.song-search.dll   /kaggle/working/matcool.song-search.dll 2>/dev/null || true
fi
cp -f /kaggle/working/BUILD-LOG.txt /kaggle/working/song-search/BUILD-LOG.txt 2>/dev/null || true
rm -rf build/_deps
rm -rf build/bindings
rm -f  build/bindings/*.lib build/*.exp build/*.ilk build/CMakeFiles/*/*.obj 2>/dev/null || true
du -sh /kaggle/working/song-search /kaggle/working/*.geode 2>/dev/null | tee -a "$LOG"
log "ALL DONE"
